In [ ]:
a=["op_unique_carrier","op_carrier" , "fl_date" , "crs_dep_time" , "crs_arr_time" , "origin_airport_id" ,"dest_airport_id" , "distance" ,"tail_num",  "op_carrier_fl_num"]

In [ ]:
a = ["op_unique_carrier", "op_carrier", "fl_date", "crs_dep_time", "crs_arr_time",
     "origin_airport_id", "dest_airport_id", "distance", "tail_num", "op_carrier_fl_num"]

import pandas as pd
from sqlalchemy import create_engine, func, select
from sqlalchemy.orm import sessionmaker
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os

load_dotenv()

user = os.getenv('user')
password = os.getenv('password')
host = os.getenv('host')
port = os.getenv('port')
dbname = os.getenv('dbname')
db_url = f"postgresql://{user}:{password}@{host}:{port}/{dbname}"

# Definición del modelo
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy import Column, Integer, String, Float, Date

Base = declarative_base()

class Flight(Base):
    __tablename__ = 'flights'
    
    id = Column(Integer, primary_key=True)
    fl_date = Column(Date)
    arr_delay = Column(Float)

# Conexión y sesión
engine = create_engine(db_url)
Session = sessionmaker(bind=engine)
session = Session()

# Obtener rango de fechas
max_date = session.query(func.max(Flight.fl_date)).scalar()
one_yearld = max_date - timedelta(days=365)

# Añadimos 'arr_delay' a la lista de columnas a seleccionar
columns_str = ", ".join(a + ['arr_delay'])

# Usamos text() correctamente
from sqlalchemy import text

query = session.query(text(columns_str)).filter(
    Flight.fl_date >= one_yearld,
    Flight.arr_delay.isnot(None)
)

# Cargar en DataFrame
flights_df = pd.read_sql(query.statement, engine)
print(len(flights_df))
session.close()

In [ ]:
print(flights_df.isnull().sum())
print(flights_df[["distance","arr_delay"]].describe())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.boxplot(flights_df['arr_delay'].dropna(), vert=True)  # `vert=True` para orientación vertical
plt.title('Distribución de arr_delay (Boxplot)')
plt.ylabel('Minutos de retraso (negativos = adelanto)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
upper = flights_df['arr_delay'].quantile(0.95)

flights_clean = flights_df[(flights_df['arr_delay'] <= upper)]

print(f"Datos limpios (percentil <95%): {len(flights_clean)} filas")

In [ ]:
print(flights_clean[["distance","arr_delay"]].describe())
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.boxplot(flights_clean['arr_delay'].dropna(), vert=True)  # `vert=True` para orientación vertical
plt.title('Distribución de arr_delay (Boxplot)')
plt.ylabel('Minutos de retraso (negativos = adelanto)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
print(a)
print(flights_df.isnull().sum())
print(flights_df[["distance","arr_delay"]].describe())
flights_df1=flights_df

In [ ]:
flights_df['crs_arr_time']
pd.to_datetime(flights_df['crs_dep_time'], format='%H:%M:%S', errors='coerce').dt.time

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from catboost import Pool
from datetime import datetime, timedelta

# 1. Selección de características MEJORADA
features = [
    "op_unique_carrier",
    "op_carrier",
    "origin_airport_id",
    "dest_airport_id",
    "distance",
    "op_carrier_fl_num",
    "day_of_week",
    "month",
    "is_weekend",
    "scheduled_duration",
    "dep_hour",
    "dep_minute",
    "arr_hour"
]

# 2. Preprocesamiento ANTES de la división
flights_df = flights_df.copy()  # Evita SettingWithCopyWarning

# Convertir fechas y extraer características temporales
flights_df['fl_date'] = pd.to_datetime(flights_df['fl_date'])
flights_df['day_of_week'] = flights_df['fl_date'].dt.dayofweek
flights_df['month'] = flights_df['fl_date'].dt.month
flights_df['is_weekend'] = flights_df['day_of_week'].isin([5,6]).astype(int)

# Convertir tiempos y calcular duración programada
flights_df['crs_arr_time'] = pd.to_datetime(flights_df['crs_arr_time'], format='%H:%M:%S', errors='coerce').dt.time
flights_df['crs_dep_time'] = pd.to_datetime(flights_df['crs_dep_time'], format='%H:%M:%S', errors='coerce').dt.time

# Extraer hora del día
flights_df['dep_hour'] = flights_df['crs_dep_time'].apply(lambda x: x.hour)
flights_df['dep_minute'] = flights_df['crs_dep_time'].apply(lambda x: x.minute)
flights_df['arr_hour'] = flights_df['crs_arr_time'].apply(lambda x: x.hour)
flights_df['arr_minute'] = flights_df['crs_arr_time'].apply(lambda x: x.minute)

def calculate_duration(row):
    dep_datetime = datetime.combine(datetime.min, row['crs_dep_time'])
    arr_datetime = datetime.combine(datetime.min, row['crs_arr_time'])
    if arr_datetime < dep_datetime:
        arr_datetime += timedelta(days=1)  # Para vuelos que cruzan la medianoche
    return (arr_datetime - dep_datetime).total_seconds() / 60

flights_df['scheduled_duration'] = flights_df.apply(calculate_duration, axis=1)

X = flights_df[features]
y = flights_df["arr_delay"]  # Mejor como Series que como DataFrame para CatBoost

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=flights_df['month'])
X_test,xtest, y_test, ytest = train_test_split(X, y, test_size=0.2, random_state=42, stratify=flights_df['month'])

cat_features = [
    "op_unique_carrier",
    "op_carrier",
    "op_carrier_fl_num",
    "origin_airport_id",
    "dest_airport_id",
    "day_of_week",
    "month",
    "dep_hour",
    "arr_hour"
]

# 7. Creación de pools (correcto como lo tenías)
train_pool = Pool(data=X_train, label=y_train, cat_features=cat_features)
test_pool = Pool(data=X_test, label=y_test, cat_features=cat_features)

In [ ]:
X_test,xtest, y_test, ytest = train_test_split(X, y, test_size=0.2, random_state=42, stratify=flights_df['month'])
test_pool_1 = Pool(data=xtest, label=ytest, cat_features=cat_features)
print(len(xtest),len(ytest))

In [ ]:
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import torch

model = CatBoostRegressor(
    iterations=1250,  
    learning_rate=0.08,  # Tasa de aprendizaje más conservadora
    depth=8,  # Mayor profundidad para capturar relaciones más complejas
    l2_leaf_reg=3,  # Regularización L2 para prevenir overfitting
    random_seed=42,
    eval_metric='MAE',
    loss_function='MAE', 
    verbose=100
)

model.fit(train_pool,eval_set=test_pool,use_best_model=True,plot=True )

# Predicción CORRECTA usando test_pool (no X_test directamente)
preds = model.predict(test_pool)
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print(f"\nResultados en Test:")
print(f"MAE: {mae:.2f} minutos")
print(f"RMSE: {rmse:.2f} minutos")
print(f"R²: {r2:.4f}")

# Feature importance
feature_importance = model.get_feature_importance(prettified=True)
print("\nImportancia de características:")
print(feature_importance)

# Guardar el modelo para uso futuro
model.save_model('flight_delay_model.cbm')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import Pool

# Asumiendo que ya tienes definidos:
# - xtest: características de prueba (puede ser DataFrame o array)
# - ytest: etiquetas reales (array o serie)
# - cat_features: lista de índices de columnas categóricas
# - model: modelo entrenado de CatBoost

# 1. Crea el pool solo para predicción (esto es correcto)
test_pool = Pool(data=xtest, label=ytest, cat_features=cat_features)

# 2. Haz predicciones
preds = model.predict(test_pool)
preds_array = np.array(preds) if not isinstance(preds, np.ndarray) else preds

# 3. Asegúrate de que ytest sea un array o serie
if isinstance(ytest, pd.DataFrame) or isinstance(ytest, pd.Series):
    y_test_series = ytest.squeeze()  # Convertir a Serie si es DataFrame
else:
    y_test_series = pd.Series(ytest, name="Valor_Real")

# 4. Filtrar valores extremos (por ejemplo, percentil 95)
upper = y_test_series.quantile(0.95)
mask = y_test_series <= upper

# 5. Aplicar el filtro a ambos: y_test_series y preds_array
y_filtered = y_test_series[mask]
preds_filtered = preds_array[mask]

# 6. Crear DataFrame final
resultados_df = pd.DataFrame({
    "Valor_Real": y_filtered,
    "Estimado": preds_filtered
})

# 7. Calcular residuos y errores absolutos
resultados_df["Residuo"] = resultados_df["Valor_Real"] - resultados_df["Estimado"]
resultados_df["Error_Absoluto"] = np.abs(resultados_df["Residuo"])

# 8. Imprimir estadísticas
print("\nEstadísticas de los residuos:")
print(resultados_df["Residuo"].describe())

print("\nMuestra de predicciones vs valores reales:")
print(resultados_df.head(10).to_string(float_format="%.2f"))

# 9. Graficar
plt.figure(figsize=(12, 6))

# Histograma de residuos
plt.subplot(1, 2, 1)
sns.histplot(resultados_df["Residuo"], bins=30, kde=True, color='skyblue')
plt.title("Distribución de los Residuos")
plt.xlabel("Residuo (Real - Predicho)")
plt.ylabel("Frecuencia")
plt.axvline(0, color='red', linestyle='--', linewidth=1)

# Gráfico de dispersión Real vs Estimado
plt.subplot(1, 2, 2)
sns.scatterplot(x="Valor_Real", y="Estimado", data=resultados_df, alpha=0.6)
plt.plot([resultados_df["Valor_Real"].min(), resultados_df["Valor_Real"].max()],
         [resultados_df["Valor_Real"].min(), resultados_df["Valor_Real"].max()],
         color='red', linestyle='--')
plt.title("Valor Real vs Estimado")
plt.xlabel("Valor Real")
plt.ylabel("Valor Estimado")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(ytest, preds)
rmse = np.sqrt(mean_squared_error(ytest, preds))
r2 = r2_score(ytest, preds)

print(f"MAE (Error absoluto medio): {mae:.2f} minutos")
print(f"RMSE (Raíz del error cuadrático medio): {rmse:.2f} minutos")
print(f"R² (Coeficiente de determinación): {r2:.2f}")


In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(resultados_df["Estimado"], resultados_df["Residuo"], alpha=0.5)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Retraso predicho (minutos)")
plt.ylabel("Residuo (Error: real - predicho)")
plt.title("Gráfico de residuos")
plt.show()


Cargar modelo

In [ ]:
from catboost import CatBoostRegressor

# Crear instancia sin entrenar
modelo_cargado = CatBoostRegressor()

# Cargar los pesos
modelo_cargado.load_model("modelo_retrasos.cbm")


In [63]:
import requests
import pandas as pd
from datetime import datetime
API_KEY=API_KEY="3cc1553a0430315ba650c47fd8798996"
# URL del endpoint de vuelos en tiempo real
url = f' https://api.aviationstack.com/v1/flights?access_key={API_KEY}'

# 6. Diccionario opcional para mapear IATA a airport_id personalizado
iata_to_id_dict = {'BHM': 10599, 'LGA': 12953, 'ATL': 10397, 'MOB': 13422, 'HSV': 12217, 'IAH': 12266, 'DFW': 11298, 'ORD': 13930, 'MGM': 13277, 'MIA': 13303, 'DCA': 11278, 'CLT': 11057, 'PHL': 14100, 'DEN': 11292, 'DHN': 11308, 'DTW': 11433, 'MDW': 13232, 'FLL': 11697, 'DAL': 11259, 'TPA': 15304, 'MCO': 13204, 'BWI': 10821, 'HOU': 12191, 'LAS': 12889, 'IAD': 12264, 'BFM': 10562, 'XNA': 15919, 'MSP': 13487, 'CLL': 11049, 'PHX': 14107, 'BNA': 10693, 'OKC': 13851, 'BTR': 10781, 'IND': 12339, 'MCI': 13198, 'AUS': 10423, 'CAE': 10868, 'MSN': 13485, 'MKE': 13342, 'LAX': 12892, 'Atlanta, GA': 1039705, 'Oakland, CA': 1379603, 'Detroit, MI': 1143302, 'Kansas City, MO': 1319801, 'Las Vegas, NV': 1288903, 'San Francisco, CA': 1477102, 'New York, NY': 1247803, 'Minneapolis, MN': 1348702, 'Los Angeles, CA': 1289203, 'Newark, NJ': 1161802, 'Milwaukee, WI': 1334205, 'Chicago, IL': 1393004, 'Washington, DC': 1127803, 'Baltimore, MD': 1082103, 'Orlando, FL': 1320402, 'Cleveland, OH': 1104203, 'St. Louis, MO': 1501603, 'Columbus, GA': 1115003, 'Fort Wayne, IN': 1182304, 'Green Bay, WI': 1197702, 'Wilmington, DE': 1232002, 'Phoenix, AZ': 1410702, 'Dallas/Fort Worth, TX': 1129804, 'Charlotte, NC': 1105703, 'Philadelphia, PA': 1410002, 'Denver, CO': 1129202, 'Pittsburgh, PA': 1412202, 'Raleigh/Durham, NC': 1449202, 'Albuquerque, NM': 1014003, 'Charleston, SC': 1099402, 'Dayton, OH': 1126702, 'Tulsa, OK': 1537002, 'Jacksonville, FL': 1245102, 'San Antonio, TX': 1468303, 'Bakersfield, CA': 1056103, 'Salt Lake City, UT': 1486903, 'Portland, OR': 1405702, 'Honolulu, HI': 1217302, 'Houston, TX': 1219102, 'Miami, FL': 1330303, 'Hartford, CT': 1052904, 'Charleston/Dunbar, WV': 1114603, 'Little Rock, AR': 1299204, 'Mammoth Lakes, CA': 1338801, 'Lexington, KY': 1294503, 'Charlotte Amalie, VI': 1502403, 'Atlantic City, NJ': 1015804, 'Fort Lauderdale, FL': 1169704, 'North Bend/Coos Bay, OR': 1396403, 'Des Moines, IA': 1142303, 'Jackson/Vicksburg, MS': 1244805, 'Myrtle Beach, SC': 1357702, 'Fairbanks, AK': 1163002, 'Seattle, WA': 1474703, 'Cincinnati, OH': 1119302, 'Tampa, FL': 1530402, 'Sarasota/Bradenton, FL': 1498603, 'Oklahoma City, OK': 1385103, 'Lake Charles, LA': 1291503, 'Corpus Christi, TX': 1114005, 'Norfolk, VA': 1393102, 'Omaha, NE': 1387102, 'Birmingham, AL': 1059904, 'Grand Rapids, MI': 1198603, 'Columbus, OH': 1106603, 'Syracuse, NY': 1509602, 'Cedar Rapids/Iowa City, IA': 1100303, 'Wichita, KS': 1227803, 'Providence, RI': 1430702, 'Fort Myers, FL': 1463502, 'Boston, MA': 1072102, 'West Palm Beach/Palm Beach, FL': 1402702, 'Columbia, SC': 1086803, 'Asheville, NC': 1043103, 'Indianapolis, IN': 1233904, 'Savannah, GA': 1468502, 'Greer, SC': 1199603, 'Dallas, TX': 1125903, 'Huntsville, AL': 1221702, 'Richmond, VA': 1452401, 'Memphis, TN': 1324402, 'Newport News/Williamsburg, VA': 1409803, 'Sioux Falls, SD': 1177502, 'Harrisburg, PA': 1323002, 'Lafayette, LA': 1295104, 'Shreveport, LA': 1481402, 'Montgomery, AL': 1327702, 'Manchester, NH': 1329604, 'Springfield, MO': 1478302, 'Knoxville, TN': 1541203, 'Augusta, GA': 1020803, 'Buffalo, NY': 1079204, 'Peoria, IL': 1410803, 'Greensboro/High Point, NC': 1199502, 'Montrose/Delta, CO': 1350202, 'Nashville, TN': 1069302, 'Durango, CO': 1141304, 'Amarillo, TX': 1027903, 'Mosinee, WI': 1120302, 'Fargo, ND': 1163703, 'Trenton, NJ': 1535602, 'Grand Junction, CO': 1192102, 'Madison, WI': 1348502, 'San Diego, CA': 1467903, 'New Orleans, LA': 1349503, 'Akron, OH': 1087402, 'Kahului, HI': 1383002, 'Mobile, AL': 1342202, 'Dubuque, IA': 1127402, 'San Angelo, TX': 1484202, 'Boise, ID': 1071302, 'Rochester, NY': 1457604, 'Spokane, WA': 1188402, 'San Jose, CA': 1483103, 'Burbank, CA': 1080003, 'El Paso, TX': 1154003, 'Ontario, CA': 1389101, 'Reno, NV': 1457002, 'Santa Ana, CA': 1490803, 'Austin, TX': 1042302, 'Albany, NY': 1025702, 'Long Beach, CA': 1295402, 'Louisville, KY': 1473003, 'Sacramento, CA': 1489302, 'Killeen, TX': 1198202, 'Fresno, CA': 1163805, 'Yuma, AZ': 1621801, 'Tucson, AZ': 1537602, 'Palm Springs, CA': 1426204, 'Jackson, WY': 1244102, 'Appleton, WI': 1040803, 'Eugene, OR': 1160302, 'Helena, MT': 1215603, 'Sun Valley/Hailey/Ketchum, ID': 1504102, 'Williston, ND': 1238902, 'Midland/Odessa, TX': 1315802, 'San Luis Obispo, CA': 1469802, 'Springfield, IL': 1495203, 'Monterey, CA': 1347603, 'Aspen, CO': 1037203, 'Bozeman, MT': 1084903, 'Waco, TX': 1015502, 'Colorado Springs, CO': 1110902, 'Santa Barbara, CA': 1468902, 'Arcata/Eureka, CA': 1015703, 'Anchorage, AK': 1029904, 'Billings, MT': 1062002, 'Roanoke, VA': 1457403, 'San Juan, PR': 1484304, 'Panama City, FL': 1148102, 'Valparaiso, FL': 1562402, 'White Plains, NY': 1219702, 'Burlington, VT': 1078502, 'Missoula, MT': 1348602, 'Pensacola, FL': 1419303, 'Lubbock, TX': 1289605, 'Hilo, HI': 1240203, 'Idaho Falls, ID': 1228002, 'Pellston, MI': 1415002, 'Medford, OR': 1326403, 'Rapid City, SD': 1445702, 'Duluth, MN': 1133703, 'Aguadilla, PR': 1073203, 'Kona, HI': 1275803, 'Baton Rouge, LA': 1078103, 'South Bend, IN': 1469605, 'Kalamazoo, MI': 1046902, 'Key West, FL': 1162402, 'Scranton/Wilkes-Barre, PA': 1043403, 'Lihue, HI': 1298202, 'Joplin, MO': 1251102, 'Tallahassee, FL': 1524904, 'Melbourne, FL': 1336003, 'Fayetteville, AR': 1591902, 'Sault Ste. Marie, MI': 1101303, 'Elko, NV': 1152503, 'Daytona Beach, FL': 1125203, 'Valdosta, GA': 1560702, 'Portland, ME': 1432103, 'Carlsbad, CA': 1104103, 'Bristol/Johnson City/Kingsport, TN': 1532302, 'Gillette, WY': 1186502, 'Crescent City, CA': 1093002, 'Great Falls, MT': 1200302, 'Brainerd, MN': 1073905, 'Hobbs, NM': 1217703, 'Gulfport/Biloxi, MS': 1197302, 'Mission/McAllen/Edinburg, TX': 1325602, 'Gainesville, FL': 1195302, 'Charlottesville, VA': 1099002, 'Hayden, CO': 1209402, 'St. Augustine, FL': 1549703, 'Manhattan/Ft. Riley, KS': 1329002, 'Minot, ND': 1343302, 'Chattanooga, TN': 1098002, 'Wilmington, NC': 1232303, 'Dothan, AL': 1130802, 'Casper, WY': 1112203, 'Muskegon, MI': 1334402, 'Bend/Redmond, OR': 1448902, 'Branson, MO': 1064301, 'Harlingen/San Benito, TX': 1220603, 'Lansing, MI': 1288403, 'Flint, MI': 1172103, 'Texarkana, AR': 1540103, 'Pasco/Kennewick/Richland, WA': 1425202, 'Modesto, CA': 1342402, 'Twin Falls, ID': 1538902, 'Nantucket, MA': 1015403, 'Pocatello, ID': 1411302, 'Redding, CA': 1448702, 'Tyler, TX': 1541103, 'Elmira/Corning, NY': 1153703, 'College Station/Bryan, TX': 1104902, 'Monroe, LA': 1337702, 'Traverse City, MI': 1538003, 'Grand Forks, ND': 1189802, 'Rock Springs, WY': 1454302, 'Jacksonville/Camp Lejeune, NC': 1379502, 'Dickinson, ND': 1131502, 'Fayetteville, NC': 1164102, 'Plattsburgh, NY': 1402501, 'Erie, PA': 1157704, 'Albany, GA': 1014602, 'Allentown/Bethlehem/Easton, PA': 1013503, 'Adak Island, AK': 1016502, 'Worcester, MA': 1393303, 'Santa Fe, NM': 1467402, 'Binghamton, NY': 1057703, 'Bloomington/Normal, IL': 1068502, 'Brunswick, GA': 1073103, 'State College, PA': 1471102, 'Moline, IL': 1336703, 'Rochester, MN': 1463303, 'Devils Lake, ND': 1144703, 'Eau Claire, WI': 1147103, 'International Falls, MN': 1234302, 'Kotzebue, AK': 1397002, 'Brownsville, TX': 1074702, 'Alexandria, LA': 1018502, 'Lincoln, NE': 1302902, 'Sioux City, IA': 1504803, 'Abilene, TX': 1013603, 'La Crosse, WI': 1307602, 'Newburgh/Poughkeepsie, NY': 1507002, 'Columbia, MO': 1111102, 'St. George, UT': 1479402, 'Fort Smith, AR': 1177801, 'Wichita Falls, TX': 1496002, 'Meridian, MS': 1324102, 'Saginaw/Bay City/Midland, MI': 1318403, 'Paducah, KY': 1400602, 'Ithaca/Cortland, NY': 1239702, 'Pago Pago, TT': 1422204, 'Garden City, KS': 1186703, 'Evansville, IN': 1161204, 'Bismarck/Mandan, ND': 1062702, 'Kalispell, MT': 1164802, 'Flagstaff, AZ': 1169502, 'Pueblo, CO': 1428803, 'Hancock/Houghton, MI': 1107602, 'Butte, MT': 1077902, 'Hattiesburg/Laurel, MS': 1410902, 'Hibbing, MN': 1212903, 'Watertown, NY': 1036102, 'Sitka, AK': 1482802, 'Chico, CA': 1100202, 'Laredo, TX': 1306104, 'Aberdeen, SD': 1014102, 'Latrobe, PA': 1289803, 'Christiansted, VI': 1502704, 'Jamestown, ND': 1251902, 'Ponce, PR': 1425403, 'Juneau, AK': 1252304, 'Beaumont/Port Arthur, TX': 1072804, 'New Bern/Morehead/Beaufort, NC': 1161706, 'Bangor, ME': 1058102, 'Grand Island, NE': 1198002, 'Alpena, MI': 1033302, 'Rhinelander, WI': 1452002, 'Vernal, UT': 1558202, 'Columbus, MS': 1200702, 'Champaign/Urbana, IL': 1106702, 'Nome, AK': 1387303, 'Bellingham, WA': 1066602, 'Lawton/Fort Sill, OK': 1289102, 'Hays, KS': 1225502, 'St. Cloud, MN': 1500802, 'Islip, NY': 1239102, 'Santa Maria, CA': 1490503, 'Bethel, AK': 1055102, 'Dillingham, AK': 1133602, 'Lewiston, ID': 1312702, 'Topeka, KS': 1172602, 'Gunnison, CO': 1201203, 'Marquette, MI': 1345902, 'Klamath Falls, OR': 1302402, 'Deadhorse, AK': 1470903, 'Ketchikan, AK': 1281902, 'Petersburg, AK': 1425603, 'Waterloo, IA': 1026802, 'Longview, TX': 1190502, 'Barrow, AK': 1075402, 'Bemidji, MN': 1063104, 'Yakutat, AK': 1599102, 'Cedar City, UT': 1091802, 'Eagle, CO': 1150303, 'Iron Mountain/Kingsfd, MI': 1233502, 'Toledo, OH': 1529502, 'Escanaba, MI': 1158702, 'Guam, TT': 1201602, 'Niagara Falls, NY': 1226503, 'Laramie, WY': 1288802, 'Cody, WY': 1109702, 'Moab, UT': 1109202, 'Gustavus, AK': 1199702, 'Roswell, NM': 1458801, 'Cordova, AK': 1092603, 'Wrangell, AK': 1584102, 'King Salmon, AK': 1024502, 'Hyannis, MA': 1225002, "Martha's Vineyard, MA": 1354102, 'Kodiak, AK': 1017001, 'West Yellowstone, MT': 1589702, 'Saipan, TT': 1495503, 'Macon, GA': 1320302}


In [64]:
import requests
import pandas as pd
from geopy.distance import geodesic


# URL del endpoint de vuelos
url = f'https://api.aviationstack.com/v1/flights?access_key={API_KEY}'

# Hacer la solicitud GET
response = requests.get(url)
response.raise_for_status()
raw_data = response.json()

if not raw_data.get('data'):
    raise ValueError("No se encontraron datos de vuelos")

flights_df = pd.DataFrame(raw_data['data'])

# Extraer campos relevantes de forma segura
flights_df['op_carrier'] = flights_df['airline'].apply(lambda x: x.get('iata') if isinstance(x, dict) else None)
flights_df['op_carrier_fl_num'] = flights_df['flight'].apply(lambda x: x.get('number') if isinstance(x, dict) else None)

flights_df['origin_airport_id'] = flights_df['departure'].apply(lambda x: x.get('iata') if isinstance(x, dict) else None)
flights_df['dest_airport_id'] = flights_df['arrival'].apply(lambda x: x.get('iata') if isinstance(x, dict) else None)

# Horarios programados

flights_df['crs_dep_time'] = pd.to_datetime(flights_df['departure'].apply(lambda x: x.get('scheduled') if isinstance(x, dict) else None))
flights_df['crs_arr_time'] = pd.to_datetime(flights_df['arrival'].apply(lambda x: x.get('scheduled') if isinstance(x, dict) else None))

# Formato HHMM
flights_df['crs_dep_time'] = flights_df['crs_dep_time'].dt.strftime('%H:%M:%S')
flights_df['crs_arr_time'] = flights_df['crs_arr_time'].dt.strftime('%H:%M:%S')

# Duración programada (en minutos)
flights_df['scheduled_duration'] = (
    (pd.to_datetime(flights_df['crs_arr_time'], format='%H%M') -
     pd.to_datetime(flights_df['crs_dep_time'], format='%H%M')).dt.total_seconds() / 60
).astype(int)

# EXTRAER COORDENADAS DE SALIDA Y LLEGADA
def get_coordinates(row, key):
    data = row.get(key, {})
    lat = data.get('latitude')
    lon = data.get('longitude')
    return (lat, lon) if lat is not None and lon is not None else None

flights_df['origin_coords'] = flights_df.apply(lambda row: get_coordinates(row, 'departure'), axis=1)
flights_df['dest_coords'] = flights_df.apply(lambda row: get_coordinates(row, 'arrival'), axis=1)

# CALCULAR DISTANCIA ENTRE COORDENADAS
def calculate_distance(row):
    
    origin = row['origin_coords']
    destination = row['dest_coords']
    if origin and destination and None not in origin and None not in destination:
        return geodesic(origin, destination).kilometers
    return None

flights_df['distance_km'] = flights_df.apply(calculate_distance, axis=1)

# Columnas finales
model_columns = [
    'op_carrier', 'op_carrier_fl_num', 'origin_airport_id',
    'dest_airport_id', 'crs_dep_time', 'crs_arr_time',
    'scheduled_duration', 'distance_km'
]

final_df = flights_df[model_columns]

# Mostrar resultados
print(final_df.head())

ValueError: unconverted data remains when parsing with format "%H%M": ":50:00", at position 0. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [65]:
import requests
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from catboost import CatBoostRegressor, Pool
import os

load_dotenv()
API_KEY = os.getenv("API_KEY")

# tu modelo entrenado
MODEL_PATH = "flight_delay_model.cbm"

# URL para vuelos programados (scheduled)
url = f"https://api.aviationstack.com/v1/flights?access_key={API_KEY}&flight_status=scheduled"

# 🚀 Obtener los datos
response = requests.get(url)
response.raise_for_status()
raw_data = response.json()

def fetch_all_airport_coords(api_key):
    base_url = f'https://api.aviationstack.com/v1/airports?access_key={api_key}'
    airport_coords = {}
    for offset in range(0, 10000, 1000):
        url = f'{base_url}&limit=1000&offset={offset}'
        response = requests.get(url)
        data = response.json()
        if data.get('data'):
            for airport in data['data']:
                iata = airport.get('iata_code')
                lat = airport.get('latitude')
                lon = airport.get('longitude')
                if iata and lat and lon:
                    airport_coords[iata] = (float(lat), float(lon))
        else:
            break
    print(f"✅ Coordenadas cargadas: {len(airport_coords)} aeropuertos")
    return airport_coords


# ================ 3. Calcular distancia ==================
def calculate_distance(row, airport_coords):
    origin_coords = airport_coords.get(row['origin_airport_id'])
    dest_coords = airport_coords.get(row['dest_airport_id'])
    if origin_coords and dest_coords:
        return geodesic(origin_coords, dest_coords).miles
    return None

if not raw_data.get('data'):
    raise ValueError("No se encontraron vuelos programados")

flights_df = pd.DataFrame(raw_data['data'])

# 🔷 Extraer columnas relevantes
flights_df['op_carrier'] = flights_df['airline'].apply(lambda x: x.get('iata') if isinstance(x, dict) else None)
flights_df['op_carrier_fl_num'] = flights_df['flight'].apply(lambda x: x.get('number') if isinstance(x, dict) else None)

flights_df['origin_airport_id'] = flights_df['departure'].apply(lambda x: x.get('iata') if isinstance(x, dict) else None)
flights_df['dest_airport_id'] = flights_df['arrival'].apply(lambda x: x.get('iata') if isinstance(x, dict) else None)

flights_df['scheduled_dep'] = pd.to_datetime(
    flights_df['departure'].apply(lambda x: x.get('scheduled') if isinstance(x, dict) else None),
    errors='coerce'
).dt.tz_localize(None)

flights_df['scheduled_arr'] = pd.to_datetime(
    flights_df['arrival'].apply(lambda x: x.get('scheduled') if isinstance(x, dict) else None),
    errors='coerce'
).dt.tz_localize(None)

# 🔷 Limpiar filas sin horarios programados
flights_df = flights_df.dropna(subset=['scheduled_dep', 'scheduled_arr'])

# 🔷 Calcular duración programada en minutos
flights_df['scheduled_duration'] = (
    (flights_df['scheduled_arr'] - flights_df['scheduled_dep']).dt.total_seconds() / 60
)

# 🔷 Features adicionales
flights_df['day_of_week'] = flights_df['scheduled_dep'].dt.dayofweek
flights_df['month'] = flights_df['scheduled_dep'].dt.month
flights_df['is_weekend'] = flights_df['day_of_week'].isin([5, 6]).astype(int)

flights_df['dep_hour'] = flights_df['scheduled_dep'].dt.hour
flights_df['dep_minute'] = flights_df['scheduled_dep'].dt.minute
flights_df['arr_hour'] = flights_df['scheduled_arr'].dt.hour
flights_df['distance'] = flights_df.apply(lambda row: calculate_distance(row, airport_coords), axis=1)
flights_df['op_unique_carrier'] = flights_df['op_carrier']

# 🔷 Seleccionar features para el modelo
model_features = [
    "op_unique_carrier","op_carrier", "op_carrier_fl_num", "origin_airport_id", "dest_airport_id",
    "distance", "scheduled_duration", "day_of_week", "month", "is_weekend",
    "dep_hour", "dep_minute", "arr_hour"
]

flights_df = flights_df.dropna(subset=model_features)

X = flights_df[model_features]

# 🔷 Cargar modelo y predecir
model = CatBoostRegressor()
model.load_model(MODEL_PATH)


cat_features=[
        "op_unique_carrier","op_carrier","op_carrier_fl_num",
        "origin_airport_id","dest_airport_id","day_of_week","month","dep_hour","arr_hour"
    ]
pool = Pool(data=X, cat_features=cat_features)

flights_df['predicted_delay'] = model.predict(pool)

# 🔷 Mostrar resultados relevantes
print(flights_df[[
    'op_carrier', 'op_carrier_fl_num', 'origin_airport_id', 'dest_airport_id',
    'scheduled_dep', 'scheduled_arr', 'scheduled_duration', 'predicted_delay'
]].head(10))


HTTPError: 429 Client Error: Too Many Requests for url: https://api.aviationstack.com/v1/flights?access_key=6b97808f02ed0c4954cd00d3145d47cc&flight_status=scheduled

In [ ]:
print(pool)

In [ ]:
flights_df['predicted_delay'] = model.predict(pool)

# 🔷 Mostrar resultados relevantes
print(flights_df[[
    'op_carrier', 'op_carrier_fl_num', 'origin_airport_id', 'dest_airport_id',
    'scheduled_dep', 'scheduled_arr', 'scheduled_duration', 'predicted_delay'
]].head(10))


In [ ]:
import requests
import pandas as pd
from geopy.distance import geodesic
from catboost import CatBoostRegressor, Pool
from datetime import datetime
from dotenv import load_dotenv
import os
load_dotenv()

API_KEY=os.getenv('API_KEY')
# Cargar variables de entorno

# URL del endpoint de vuelos en tiempo real
url = f' https://api.aviationstack.com/v1/flights?access_key={API_KEY}'

# 6. Diccionario opcional para mapear IATA a airport_id personalizado
iata_to_id_dict = {'BHM': 10599, 'LGA': 12953, 'ATL': 10397, 'MOB': 13422, 'HSV': 12217, 'IAH': 12266, 'DFW': 11298, 'ORD': 13930, 'MGM': 13277, 'MIA': 13303, 'DCA': 11278, 'CLT': 11057, 'PHL': 14100, 'DEN': 11292, 'DHN': 11308, 'DTW': 11433, 'MDW': 13232, 'FLL': 11697, 'DAL': 11259, 'TPA': 15304, 'MCO': 13204, 'BWI': 10821, 'HOU': 12191, 'LAS': 12889, 'IAD': 12264, 'BFM': 10562, 'XNA': 15919, 'MSP': 13487, 'CLL': 11049, 'PHX': 14107, 'BNA': 10693, 'OKC': 13851, 'BTR': 10781, 'IND': 12339, 'MCI': 13198, 'AUS': 10423, 'CAE': 10868, 'MSN': 13485, 'MKE': 13342, 'LAX': 12892, 'Atlanta, GA': 1039705, 'Oakland, CA': 1379603, 'Detroit, MI': 1143302, 'Kansas City, MO': 1319801, 'Las Vegas, NV': 1288903, 'San Francisco, CA': 1477102, 'New York, NY': 1247803, 'Minneapolis, MN': 1348702, 'Los Angeles, CA': 1289203, 'Newark, NJ': 1161802, 'Milwaukee, WI': 1334205, 'Chicago, IL': 1393004, 'Washington, DC': 1127803, 'Baltimore, MD': 1082103, 'Orlando, FL': 1320402, 'Cleveland, OH': 1104203, 'St. Louis, MO': 1501603, 'Columbus, GA': 1115003, 'Fort Wayne, IN': 1182304, 'Green Bay, WI': 1197702, 'Wilmington, DE': 1232002, 'Phoenix, AZ': 1410702, 'Dallas/Fort Worth, TX': 1129804, 'Charlotte, NC': 1105703, 'Philadelphia, PA': 1410002, 'Denver, CO': 1129202, 'Pittsburgh, PA': 1412202, 'Raleigh/Durham, NC': 1449202, 'Albuquerque, NM': 1014003, 'Charleston, SC': 1099402, 'Dayton, OH': 1126702, 'Tulsa, OK': 1537002, 'Jacksonville, FL': 1245102, 'San Antonio, TX': 1468303, 'Bakersfield, CA': 1056103, 'Salt Lake City, UT': 1486903, 'Portland, OR': 1405702, 'Honolulu, HI': 1217302, 'Houston, TX': 1219102, 'Miami, FL': 1330303, 'Hartford, CT': 1052904, 'Charleston/Dunbar, WV': 1114603, 'Little Rock, AR': 1299204, 'Mammoth Lakes, CA': 1338801, 'Lexington, KY': 1294503, 'Charlotte Amalie, VI': 1502403, 'Atlantic City, NJ': 1015804, 'Fort Lauderdale, FL': 1169704, 'North Bend/Coos Bay, OR': 1396403, 'Des Moines, IA': 1142303, 'Jackson/Vicksburg, MS': 1244805, 'Myrtle Beach, SC': 1357702, 'Fairbanks, AK': 1163002, 'Seattle, WA': 1474703, 'Cincinnati, OH': 1119302, 'Tampa, FL': 1530402, 'Sarasota/Bradenton, FL': 1498603, 'Oklahoma City, OK': 1385103, 'Lake Charles, LA': 1291503, 'Corpus Christi, TX': 1114005, 'Norfolk, VA': 1393102, 'Omaha, NE': 1387102, 'Birmingham, AL': 1059904, 'Grand Rapids, MI': 1198603, 'Columbus, OH': 1106603, 'Syracuse, NY': 1509602, 'Cedar Rapids/Iowa City, IA': 1100303, 'Wichita, KS': 1227803, 'Providence, RI': 1430702, 'Fort Myers, FL': 1463502, 'Boston, MA': 1072102, 'West Palm Beach/Palm Beach, FL': 1402702, 'Columbia, SC': 1086803, 'Asheville, NC': 1043103, 'Indianapolis, IN': 1233904, 'Savannah, GA': 1468502, 'Greer, SC': 1199603, 'Dallas, TX': 1125903, 'Huntsville, AL': 1221702, 'Richmond, VA': 1452401, 'Memphis, TN': 1324402, 'Newport News/Williamsburg, VA': 1409803, 'Sioux Falls, SD': 1177502, 'Harrisburg, PA': 1323002, 'Lafayette, LA': 1295104, 'Shreveport, LA': 1481402, 'Montgomery, AL': 1327702, 'Manchester, NH': 1329604, 'Springfield, MO': 1478302, 'Knoxville, TN': 1541203, 'Augusta, GA': 1020803, 'Buffalo, NY': 1079204, 'Peoria, IL': 1410803, 'Greensboro/High Point, NC': 1199502, 'Montrose/Delta, CO': 1350202, 'Nashville, TN': 1069302, 'Durango, CO': 1141304, 'Amarillo, TX': 1027903, 'Mosinee, WI': 1120302, 'Fargo, ND': 1163703, 'Trenton, NJ': 1535602, 'Grand Junction, CO': 1192102, 'Madison, WI': 1348502, 'San Diego, CA': 1467903, 'New Orleans, LA': 1349503, 'Akron, OH': 1087402, 'Kahului, HI': 1383002, 'Mobile, AL': 1342202, 'Dubuque, IA': 1127402, 'San Angelo, TX': 1484202, 'Boise, ID': 1071302, 'Rochester, NY': 1457604, 'Spokane, WA': 1188402, 'San Jose, CA': 1483103, 'Burbank, CA': 1080003, 'El Paso, TX': 1154003, 'Ontario, CA': 1389101, 'Reno, NV': 1457002, 'Santa Ana, CA': 1490803, 'Austin, TX': 1042302, 'Albany, NY': 1025702, 'Long Beach, CA': 1295402, 'Louisville, KY': 1473003, 'Sacramento, CA': 1489302, 'Killeen, TX': 1198202, 'Fresno, CA': 1163805, 'Yuma, AZ': 1621801, 'Tucson, AZ': 1537602, 'Palm Springs, CA': 1426204, 'Jackson, WY': 1244102, 'Appleton, WI': 1040803, 'Eugene, OR': 1160302, 'Helena, MT': 1215603, 'Sun Valley/Hailey/Ketchum, ID': 1504102, 'Williston, ND': 1238902, 'Midland/Odessa, TX': 1315802, 'San Luis Obispo, CA': 1469802, 'Springfield, IL': 1495203, 'Monterey, CA': 1347603, 'Aspen, CO': 1037203, 'Bozeman, MT': 1084903, 'Waco, TX': 1015502, 'Colorado Springs, CO': 1110902, 'Santa Barbara, CA': 1468902, 'Arcata/Eureka, CA': 1015703, 'Anchorage, AK': 1029904, 'Billings, MT': 1062002, 'Roanoke, VA': 1457403, 'San Juan, PR': 1484304, 'Panama City, FL': 1148102, 'Valparaiso, FL': 1562402, 'White Plains, NY': 1219702, 'Burlington, VT': 1078502, 'Missoula, MT': 1348602, 'Pensacola, FL': 1419303, 'Lubbock, TX': 1289605, 'Hilo, HI': 1240203, 'Idaho Falls, ID': 1228002, 'Pellston, MI': 1415002, 'Medford, OR': 1326403, 'Rapid City, SD': 1445702, 'Duluth, MN': 1133703, 'Aguadilla, PR': 1073203, 'Kona, HI': 1275803, 'Baton Rouge, LA': 1078103, 'South Bend, IN': 1469605, 'Kalamazoo, MI': 1046902, 'Key West, FL': 1162402, 'Scranton/Wilkes-Barre, PA': 1043403, 'Lihue, HI': 1298202, 'Joplin, MO': 1251102, 'Tallahassee, FL': 1524904, 'Melbourne, FL': 1336003, 'Fayetteville, AR': 1591902, 'Sault Ste. Marie, MI': 1101303, 'Elko, NV': 1152503, 'Daytona Beach, FL': 1125203, 'Valdosta, GA': 1560702, 'Portland, ME': 1432103, 'Carlsbad, CA': 1104103, 'Bristol/Johnson City/Kingsport, TN': 1532302, 'Gillette, WY': 1186502, 'Crescent City, CA': 1093002, 'Great Falls, MT': 1200302, 'Brainerd, MN': 1073905, 'Hobbs, NM': 1217703, 'Gulfport/Biloxi, MS': 1197302, 'Mission/McAllen/Edinburg, TX': 1325602, 'Gainesville, FL': 1195302, 'Charlottesville, VA': 1099002, 'Hayden, CO': 1209402, 'St. Augustine, FL': 1549703, 'Manhattan/Ft. Riley, KS': 1329002, 'Minot, ND': 1343302, 'Chattanooga, TN': 1098002, 'Wilmington, NC': 1232303, 'Dothan, AL': 1130802, 'Casper, WY': 1112203, 'Muskegon, MI': 1334402, 'Bend/Redmond, OR': 1448902, 'Branson, MO': 1064301, 'Harlingen/San Benito, TX': 1220603, 'Lansing, MI': 1288403, 'Flint, MI': 1172103, 'Texarkana, AR': 1540103, 'Pasco/Kennewick/Richland, WA': 1425202, 'Modesto, CA': 1342402, 'Twin Falls, ID': 1538902, 'Nantucket, MA': 1015403, 'Pocatello, ID': 1411302, 'Redding, CA': 1448702, 'Tyler, TX': 1541103, 'Elmira/Corning, NY': 1153703, 'College Station/Bryan, TX': 1104902, 'Monroe, LA': 1337702, 'Traverse City, MI': 1538003, 'Grand Forks, ND': 1189802, 'Rock Springs, WY': 1454302, 'Jacksonville/Camp Lejeune, NC': 1379502, 'Dickinson, ND': 1131502, 'Fayetteville, NC': 1164102, 'Plattsburgh, NY': 1402501, 'Erie, PA': 1157704, 'Albany, GA': 1014602, 'Allentown/Bethlehem/Easton, PA': 1013503, 'Adak Island, AK': 1016502, 'Worcester, MA': 1393303, 'Santa Fe, NM': 1467402, 'Binghamton, NY': 1057703, 'Bloomington/Normal, IL': 1068502, 'Brunswick, GA': 1073103, 'State College, PA': 1471102, 'Moline, IL': 1336703, 'Rochester, MN': 1463303, 'Devils Lake, ND': 1144703, 'Eau Claire, WI': 1147103, 'International Falls, MN': 1234302, 'Kotzebue, AK': 1397002, 'Brownsville, TX': 1074702, 'Alexandria, LA': 1018502, 'Lincoln, NE': 1302902, 'Sioux City, IA': 1504803, 'Abilene, TX': 1013603, 'La Crosse, WI': 1307602, 'Newburgh/Poughkeepsie, NY': 1507002, 'Columbia, MO': 1111102, 'St. George, UT': 1479402, 'Fort Smith, AR': 1177801, 'Wichita Falls, TX': 1496002, 'Meridian, MS': 1324102, 'Saginaw/Bay City/Midland, MI': 1318403, 'Paducah, KY': 1400602, 'Ithaca/Cortland, NY': 1239702, 'Pago Pago, TT': 1422204, 'Garden City, KS': 1186703, 'Evansville, IN': 1161204, 'Bismarck/Mandan, ND': 1062702, 'Kalispell, MT': 1164802, 'Flagstaff, AZ': 1169502, 'Pueblo, CO': 1428803, 'Hancock/Houghton, MI': 1107602, 'Butte, MT': 1077902, 'Hattiesburg/Laurel, MS': 1410902, 'Hibbing, MN': 1212903, 'Watertown, NY': 1036102, 'Sitka, AK': 1482802, 'Chico, CA': 1100202, 'Laredo, TX': 1306104, 'Aberdeen, SD': 1014102, 'Latrobe, PA': 1289803, 'Christiansted, VI': 1502704, 'Jamestown, ND': 1251902, 'Ponce, PR': 1425403, 'Juneau, AK': 1252304, 'Beaumont/Port Arthur, TX': 1072804, 'New Bern/Morehead/Beaufort, NC': 1161706, 'Bangor, ME': 1058102, 'Grand Island, NE': 1198002, 'Alpena, MI': 1033302, 'Rhinelander, WI': 1452002, 'Vernal, UT': 1558202, 'Columbus, MS': 1200702, 'Champaign/Urbana, IL': 1106702, 'Nome, AK': 1387303, 'Bellingham, WA': 1066602, 'Lawton/Fort Sill, OK': 1289102, 'Hays, KS': 1225502, 'St. Cloud, MN': 1500802, 'Islip, NY': 1239102, 'Santa Maria, CA': 1490503, 'Bethel, AK': 1055102, 'Dillingham, AK': 1133602, 'Lewiston, ID': 1312702, 'Topeka, KS': 1172602, 'Gunnison, CO': 1201203, 'Marquette, MI': 1345902, 'Klamath Falls, OR': 1302402, 'Deadhorse, AK': 1470903, 'Ketchikan, AK': 1281902, 'Petersburg, AK': 1425603, 'Waterloo, IA': 1026802, 'Longview, TX': 1190502, 'Barrow, AK': 1075402, 'Bemidji, MN': 1063104, 'Yakutat, AK': 1599102, 'Cedar City, UT': 1091802, 'Eagle, CO': 1150303, 'Iron Mountain/Kingsfd, MI': 1233502, 'Toledo, OH': 1529502, 'Escanaba, MI': 1158702, 'Guam, TT': 1201602, 'Niagara Falls, NY': 1226503, 'Laramie, WY': 1288802, 'Cody, WY': 1109702, 'Moab, UT': 1109202, 'Gustavus, AK': 1199702, 'Roswell, NM': 1458801, 'Cordova, AK': 1092603, 'Wrangell, AK': 1584102, 'King Salmon, AK': 1024502, 'Hyannis, MA': 1225002, "Martha's Vineyard, MA": 1354102, 'Kodiak, AK': 1017001, 'West Yellowstone, MT': 1589702, 'Saipan, TT': 1495503, 'Macon, GA': 1320302}

MODEL_PATH = "/home/jennifer/Documentos/tercer_año/segundo_semestre/ID/AeroData/app/predictions/flight_delay_model.cbm"

# =============== 1. Cargar datos de vuelos ==================
def fetch_flight_data():
    url = f'https://api.aviationstack.com/v1/flights?access_key={API_KEY}&flight_status=landed'
    response = requests.get(url)
    response.raise_for_status()
    raw_data = response.json()
    if not raw_data.get('data'):
        raise ValueError("No se encontraron datos de vuelos")
    return pd.DataFrame(raw_data['data'])


def extract_flight_fields(flights_df):
    flights_df['op_carrier'] = flights_df['airline'].apply(lambda x: x.get('iata') if isinstance(x, dict) else None)
    flights_df['op_carrier_fl_num'] = flights_df['flight'].apply(lambda x: x.get('number') if isinstance(x, dict) else None)

    flights_df['origin_airport_id'] = flights_df['departure'].apply(lambda x: x.get('iata') if isinstance(x, dict) else None)
    flights_df['dest_airport_id'] = flights_df['arrival'].apply(lambda x: x.get('iata') if isinstance(x, dict) else None)

    # Tiempos programados y reales
    flights_df['scheduled_dep_time'] = pd.to_datetime(flights_df['departure'].apply(lambda x: x.get('scheduled'))).dt.strftime('%H:%M:%S')
    flights_df['scheduled_arr_time'] = pd.to_datetime(flights_df['arrival'].apply(lambda x: x.get('scheduled'))).dt.strftime('%H:%M:%S')

    flights_df['actual_dep_time'] = pd.to_datetime(flights_df['departure'].apply(lambda x: x.get('actual')))
    flights_df['actual_arr_time'] = pd.to_datetime(flights_df['arrival'].apply(lambda x: x.get('actual')))

    flights_df['actual_dep_time'] = flights_df['actual_dep_time'].dt.strftime('%H:%M:%S')
    flights_df['actual_arr_time'] = flights_df['actual_arr_time'].dt.strftime('%H:%M:%S')

    return flights_df


# ================ 2. Cargar aeropuertos y coordenadas ==================
def fetch_all_airport_coords(api_key):
    base_url = f'https://api.aviationstack.com/v1/airports?access_key={api_key}'
    airport_coords = {}
    for offset in range(0, 10000, 1000):
        url = f'{base_url}&limit=1000&offset={offset}'
        response = requests.get(url)
        data = response.json()
        if data.get('data'):
            for airport in data['data']:
                iata = airport.get('iata_code')
                lat = airport.get('latitude')
                lon = airport.get('longitude')
                if iata and lat and lon:
                    airport_coords[iata] = (float(lat), float(lon))
        else:
            break
    print(f"✅ Coordenadas cargadas: {len(airport_coords)} aeropuertos")
    return airport_coords


# ================ 3. Calcular distancia ==================
def calculate_distance(row, airport_coords):
    origin_coords = airport_coords.get(row['origin_airport_id'])
    dest_coords = airport_coords.get(row['dest_airport_id'])
    if origin_coords and dest_coords:
        return geodesic(origin_coords, dest_coords).miles
    return None


# ================ 4. Validar retraso ==================
def calculate_delay(row):
    try:
        scheduled = datetime.strptime(row['scheduled_arr_time'], '%H:%M:%S')
        actual = datetime.strptime(row['actual_arr_time'], '%H:%M:%S')
        delay_minutes = (actual - scheduled).total_seconds() / 60
        return delay_minutes
    except Exception:
        return None

def get_or_create_airport_id(iata_code):
    global current_id
    if pd.isna(iata_code) or iata_code not in airport_coords:
        return None
    
    if iata_code in iata_to_id_dict:
        return iata_to_id_dict[iata_code]
    else:
        current_id += 1
        iata_to_id_dict[iata_code] = str(current_id)
        return str(current_id)
    
def calculate_duration(row):
    dep_time = datetime.strptime(row['scheduled_dep_time'], '%H:%M:%S').time()
    arr_time = datetime.strptime(row['scheduled_arr_time'], '%H:%M:%S').time()

    dep_datetime = datetime.combine(datetime.min, dep_time)
    arr_datetime = datetime.combine(datetime.min, arr_time)

    if arr_datetime < dep_datetime:
        arr_datetime += timedelta(days=1)  # Para vuelos que cruzan la medianoche
    return (arr_datetime - dep_datetime).total_seconds() / 60

# ================ 5. Preparar y predecir ==================
def prepare_for_validation(flight_df, predictions):
    flight_df = flight_df.copy()
    flight_df['predicted_delay'] = predictions
    flight_df['actual_delay'] = flight_df.apply(calculate_delay, axis=1)
    return flight_df


# ================= MAIN ==================
flights_df = fetch_flight_data()
flights_df = extract_flight_fields(flights_df)

airport_coords = fetch_all_airport_coords(API_KEY)

flights_df['distance'] = flights_df.apply(lambda row: calculate_distance(row, airport_coords), axis=1)

# Aquí puedes añadir más features según tu modelo
flights_df['day_of_week'] = datetime.today().weekday()
flights_df['month'] = datetime.today().month
flights_df['is_weekend'] = flights_df['day_of_week'].isin([5,6]).astype(int)

# dummy para op_unique_carrier (ajusta si necesario)
flights_df['op_unique_carrier'] = flights_df['op_carrier']
flights_df['scheduled_duration'] = flights_df.apply(calculate_duration, axis=1)
flights_df['dep_hour'] = flights_df['scheduled_dep_time'].apply(lambda x: int(x.split(':')[0]) if pd.notnull(x) else None)
flights_df['dep_minute'] = flights_df['scheduled_dep_time'].apply(lambda x: int(x.split(':')[1]) if pd.notnull(x) else None)
flights_df['arr_hour'] = flights_df['scheduled_arr_time'].apply(lambda x: int(x.split(':')[0]) if pd.notnull(x) else None)


current_id = 1039705

flights_df['origin_airport_id'] = flights_df['origin_airport_id'].apply(get_or_create_airport_id)
flights_df['dest_airport_id'] = flights_df['dest_airport_id'].apply(get_or_create_airport_id)

model_features = [
    "op_unique_carrier","op_carrier","origin_airport_id","dest_airport_id",
    "distance","op_carrier_fl_num","day_of_week","month","is_weekend",
    "scheduled_duration","dep_hour","dep_minute","arr_hour"
]

flights_df = flights_df.dropna(subset=model_features)
features = flights_df[model_features]
# ================ PREDICCIÓN ==================
model = CatBoostRegressor()
model.load_model(MODEL_PATH)

prediction_pool = Pool(
    data=features,
    cat_features=[
        "op_unique_carrier","op_carrier","op_carrier_fl_num",
        "origin_airport_id","dest_airport_id","day_of_week","month","dep_hour","arr_hour"
    ]
)

predictions = model.predict(prediction_pool)

# VALIDACIÓN
validation_data = prepare_for_validation(flights_df, predictions)

print(validation_data[[
    'op_carrier', 'op_carrier_fl_num', 'origin_airport_id', 'dest_airport_id',
    'scheduled_arr_time', 'actual_arr_time', 'actual_delay', 'predicted_delay'
]].head())


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

validation_data_clean = validation_data.dropna(subset=['actual_delay', 'predicted_delay'], how='any')
y_true = validation_data_clean['actual_delay']
y_pred = validation_data_clean['predicted_delay']

mae = mean_absolute_error(y_true, y_pred)
rmse = mean_squared_error(y_true, y_pred, squared=False)
r2 = r2_score(y_true, y_pred)

print(f"📊 Métricas de Predicción:")
print(f"  MAE:  {mae:.2f} minutos")
print(f"  RMSE: {rmse:.2f} minutos")
print(f"  R²:   {r2:.2f}")

# 2️⃣ Histograma de retrasos reales y predichos
plt.figure(figsize=(10,6))
sns.histplot(y_true, color='blue', label='Actual', kde=True, stat='density')
sns.histplot(y_pred, color='orange', label='Predicho', kde=True, stat='density')
plt.xlabel("Retraso (minutos)")
plt.title("Distribución de Retrasos: Real vs Predicho")
plt.legend()
plt.show()

# 3️⃣ Scatter plot: Real vs Predicho
plt.figure(figsize=(8,8))
sns.scatterplot(x=y_true, y=y_pred, alpha=0.5)
plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--') # línea perfecta
plt.xlabel("Retraso Real (min)")
plt.ylabel("Retraso Predicho (min)")
plt.title("Retrasos: Real vs Predicho")
plt.show()

# 4️⃣ Boxplot por día de la semana
plt.figure(figsize=(12,6))
sns.boxplot(x=validation_data['day_of_week'], y=validation_data['actual_delay'])
plt.xlabel("Día de la semana (0=Lunes)")
plt.ylabel("Retraso Real (min)")
plt.title("Retrasos por Día de la Semana")
plt.show()

# 5️⃣ Heatmap origen-destino vs media de retraso real
pivot_table = validation_data.pivot_table(
    index='origin_airport_id', columns='dest_airport_id',
    values='actual_delay', aggfunc='mean'
)



In [ ]:
validation_data_clean = validation_data.dropna(subset=['actual_delay', 'predicted_delay'], how='any')
print(validation_data_clean.isnull().sum())


In [ ]:
import requests
import pandas as pd
from geopy.distance import geodesic
from catboost import CatBoostRegressor, Pool
from datetime import datetime
from dotenv import load_dotenv
import os
load_dotenv()

API_KEY=os.getenv('API_KEY')
# Cargar variables de entorno

# URL del endpoint de vuelos en tiempo real
url = f' https://api.aviationstack.com/v1/flights?access_key={API_KEY}'

# 6. Diccionario opcional para mapear IATA a airport_id personalizado
iata_to_id_dict = {'BHM': 10599, 'LGA': 12953, 'ATL': 10397, 'MOB': 13422, 'HSV': 12217, 'IAH': 12266, 'DFW': 11298, 'ORD': 13930, 'MGM': 13277, 'MIA': 13303, 'DCA': 11278, 'CLT': 11057, 'PHL': 14100, 'DEN': 11292, 'DHN': 11308, 'DTW': 11433, 'MDW': 13232, 'FLL': 11697, 'DAL': 11259, 'TPA': 15304, 'MCO': 13204, 'BWI': 10821, 'HOU': 12191, 'LAS': 12889, 'IAD': 12264, 'BFM': 10562, 'XNA': 15919, 'MSP': 13487, 'CLL': 11049, 'PHX': 14107, 'BNA': 10693, 'OKC': 13851, 'BTR': 10781, 'IND': 12339, 'MCI': 13198, 'AUS': 10423, 'CAE': 10868, 'MSN': 13485, 'MKE': 13342, 'LAX': 12892, 'Atlanta, GA': 1039705, 'Oakland, CA': 1379603, 'Detroit, MI': 1143302, 'Kansas City, MO': 1319801, 'Las Vegas, NV': 1288903, 'San Francisco, CA': 1477102, 'New York, NY': 1247803, 'Minneapolis, MN': 1348702, 'Los Angeles, CA': 1289203, 'Newark, NJ': 1161802, 'Milwaukee, WI': 1334205, 'Chicago, IL': 1393004, 'Washington, DC': 1127803, 'Baltimore, MD': 1082103, 'Orlando, FL': 1320402, 'Cleveland, OH': 1104203, 'St. Louis, MO': 1501603, 'Columbus, GA': 1115003, 'Fort Wayne, IN': 1182304, 'Green Bay, WI': 1197702, 'Wilmington, DE': 1232002, 'Phoenix, AZ': 1410702, 'Dallas/Fort Worth, TX': 1129804, 'Charlotte, NC': 1105703, 'Philadelphia, PA': 1410002, 'Denver, CO': 1129202, 'Pittsburgh, PA': 1412202, 'Raleigh/Durham, NC': 1449202, 'Albuquerque, NM': 1014003, 'Charleston, SC': 1099402, 'Dayton, OH': 1126702, 'Tulsa, OK': 1537002, 'Jacksonville, FL': 1245102, 'San Antonio, TX': 1468303, 'Bakersfield, CA': 1056103, 'Salt Lake City, UT': 1486903, 'Portland, OR': 1405702, 'Honolulu, HI': 1217302, 'Houston, TX': 1219102, 'Miami, FL': 1330303, 'Hartford, CT': 1052904, 'Charleston/Dunbar, WV': 1114603, 'Little Rock, AR': 1299204, 'Mammoth Lakes, CA': 1338801, 'Lexington, KY': 1294503, 'Charlotte Amalie, VI': 1502403, 'Atlantic City, NJ': 1015804, 'Fort Lauderdale, FL': 1169704, 'North Bend/Coos Bay, OR': 1396403, 'Des Moines, IA': 1142303, 'Jackson/Vicksburg, MS': 1244805, 'Myrtle Beach, SC': 1357702, 'Fairbanks, AK': 1163002, 'Seattle, WA': 1474703, 'Cincinnati, OH': 1119302, 'Tampa, FL': 1530402, 'Sarasota/Bradenton, FL': 1498603, 'Oklahoma City, OK': 1385103, 'Lake Charles, LA': 1291503, 'Corpus Christi, TX': 1114005, 'Norfolk, VA': 1393102, 'Omaha, NE': 1387102, 'Birmingham, AL': 1059904, 'Grand Rapids, MI': 1198603, 'Columbus, OH': 1106603, 'Syracuse, NY': 1509602, 'Cedar Rapids/Iowa City, IA': 1100303, 'Wichita, KS': 1227803, 'Providence, RI': 1430702, 'Fort Myers, FL': 1463502, 'Boston, MA': 1072102, 'West Palm Beach/Palm Beach, FL': 1402702, 'Columbia, SC': 1086803, 'Asheville, NC': 1043103, 'Indianapolis, IN': 1233904, 'Savannah, GA': 1468502, 'Greer, SC': 1199603, 'Dallas, TX': 1125903, 'Huntsville, AL': 1221702, 'Richmond, VA': 1452401, 'Memphis, TN': 1324402, 'Newport News/Williamsburg, VA': 1409803, 'Sioux Falls, SD': 1177502, 'Harrisburg, PA': 1323002, 'Lafayette, LA': 1295104, 'Shreveport, LA': 1481402, 'Montgomery, AL': 1327702, 'Manchester, NH': 1329604, 'Springfield, MO': 1478302, 'Knoxville, TN': 1541203, 'Augusta, GA': 1020803, 'Buffalo, NY': 1079204, 'Peoria, IL': 1410803, 'Greensboro/High Point, NC': 1199502, 'Montrose/Delta, CO': 1350202, 'Nashville, TN': 1069302, 'Durango, CO': 1141304, 'Amarillo, TX': 1027903, 'Mosinee, WI': 1120302, 'Fargo, ND': 1163703, 'Trenton, NJ': 1535602, 'Grand Junction, CO': 1192102, 'Madison, WI': 1348502, 'San Diego, CA': 1467903, 'New Orleans, LA': 1349503, 'Akron, OH': 1087402, 'Kahului, HI': 1383002, 'Mobile, AL': 1342202, 'Dubuque, IA': 1127402, 'San Angelo, TX': 1484202, 'Boise, ID': 1071302, 'Rochester, NY': 1457604, 'Spokane, WA': 1188402, 'San Jose, CA': 1483103, 'Burbank, CA': 1080003, 'El Paso, TX': 1154003, 'Ontario, CA': 1389101, 'Reno, NV': 1457002, 'Santa Ana, CA': 1490803, 'Austin, TX': 1042302, 'Albany, NY': 1025702, 'Long Beach, CA': 1295402, 'Louisville, KY': 1473003, 'Sacramento, CA': 1489302, 'Killeen, TX': 1198202, 'Fresno, CA': 1163805, 'Yuma, AZ': 1621801, 'Tucson, AZ': 1537602, 'Palm Springs, CA': 1426204, 'Jackson, WY': 1244102, 'Appleton, WI': 1040803, 'Eugene, OR': 1160302, 'Helena, MT': 1215603, 'Sun Valley/Hailey/Ketchum, ID': 1504102, 'Williston, ND': 1238902, 'Midland/Odessa, TX': 1315802, 'San Luis Obispo, CA': 1469802, 'Springfield, IL': 1495203, 'Monterey, CA': 1347603, 'Aspen, CO': 1037203, 'Bozeman, MT': 1084903, 'Waco, TX': 1015502, 'Colorado Springs, CO': 1110902, 'Santa Barbara, CA': 1468902, 'Arcata/Eureka, CA': 1015703, 'Anchorage, AK': 1029904, 'Billings, MT': 1062002, 'Roanoke, VA': 1457403, 'San Juan, PR': 1484304, 'Panama City, FL': 1148102, 'Valparaiso, FL': 1562402, 'White Plains, NY': 1219702, 'Burlington, VT': 1078502, 'Missoula, MT': 1348602, 'Pensacola, FL': 1419303, 'Lubbock, TX': 1289605, 'Hilo, HI': 1240203, 'Idaho Falls, ID': 1228002, 'Pellston, MI': 1415002, 'Medford, OR': 1326403, 'Rapid City, SD': 1445702, 'Duluth, MN': 1133703, 'Aguadilla, PR': 1073203, 'Kona, HI': 1275803, 'Baton Rouge, LA': 1078103, 'South Bend, IN': 1469605, 'Kalamazoo, MI': 1046902, 'Key West, FL': 1162402, 'Scranton/Wilkes-Barre, PA': 1043403, 'Lihue, HI': 1298202, 'Joplin, MO': 1251102, 'Tallahassee, FL': 1524904, 'Melbourne, FL': 1336003, 'Fayetteville, AR': 1591902, 'Sault Ste. Marie, MI': 1101303, 'Elko, NV': 1152503, 'Daytona Beach, FL': 1125203, 'Valdosta, GA': 1560702, 'Portland, ME': 1432103, 'Carlsbad, CA': 1104103, 'Bristol/Johnson City/Kingsport, TN': 1532302, 'Gillette, WY': 1186502, 'Crescent City, CA': 1093002, 'Great Falls, MT': 1200302, 'Brainerd, MN': 1073905, 'Hobbs, NM': 1217703, 'Gulfport/Biloxi, MS': 1197302, 'Mission/McAllen/Edinburg, TX': 1325602, 'Gainesville, FL': 1195302, 'Charlottesville, VA': 1099002, 'Hayden, CO': 1209402, 'St. Augustine, FL': 1549703, 'Manhattan/Ft. Riley, KS': 1329002, 'Minot, ND': 1343302, 'Chattanooga, TN': 1098002, 'Wilmington, NC': 1232303, 'Dothan, AL': 1130802, 'Casper, WY': 1112203, 'Muskegon, MI': 1334402, 'Bend/Redmond, OR': 1448902, 'Branson, MO': 1064301, 'Harlingen/San Benito, TX': 1220603, 'Lansing, MI': 1288403, 'Flint, MI': 1172103, 'Texarkana, AR': 1540103, 'Pasco/Kennewick/Richland, WA': 1425202, 'Modesto, CA': 1342402, 'Twin Falls, ID': 1538902, 'Nantucket, MA': 1015403, 'Pocatello, ID': 1411302, 'Redding, CA': 1448702, 'Tyler, TX': 1541103, 'Elmira/Corning, NY': 1153703, 'College Station/Bryan, TX': 1104902, 'Monroe, LA': 1337702, 'Traverse City, MI': 1538003, 'Grand Forks, ND': 1189802, 'Rock Springs, WY': 1454302, 'Jacksonville/Camp Lejeune, NC': 1379502, 'Dickinson, ND': 1131502, 'Fayetteville, NC': 1164102, 'Plattsburgh, NY': 1402501, 'Erie, PA': 1157704, 'Albany, GA': 1014602, 'Allentown/Bethlehem/Easton, PA': 1013503, 'Adak Island, AK': 1016502, 'Worcester, MA': 1393303, 'Santa Fe, NM': 1467402, 'Binghamton, NY': 1057703, 'Bloomington/Normal, IL': 1068502, 'Brunswick, GA': 1073103, 'State College, PA': 1471102, 'Moline, IL': 1336703, 'Rochester, MN': 1463303, 'Devils Lake, ND': 1144703, 'Eau Claire, WI': 1147103, 'International Falls, MN': 1234302, 'Kotzebue, AK': 1397002, 'Brownsville, TX': 1074702, 'Alexandria, LA': 1018502, 'Lincoln, NE': 1302902, 'Sioux City, IA': 1504803, 'Abilene, TX': 1013603, 'La Crosse, WI': 1307602, 'Newburgh/Poughkeepsie, NY': 1507002, 'Columbia, MO': 1111102, 'St. George, UT': 1479402, 'Fort Smith, AR': 1177801, 'Wichita Falls, TX': 1496002, 'Meridian, MS': 1324102, 'Saginaw/Bay City/Midland, MI': 1318403, 'Paducah, KY': 1400602, 'Ithaca/Cortland, NY': 1239702, 'Pago Pago, TT': 1422204, 'Garden City, KS': 1186703, 'Evansville, IN': 1161204, 'Bismarck/Mandan, ND': 1062702, 'Kalispell, MT': 1164802, 'Flagstaff, AZ': 1169502, 'Pueblo, CO': 1428803, 'Hancock/Houghton, MI': 1107602, 'Butte, MT': 1077902, 'Hattiesburg/Laurel, MS': 1410902, 'Hibbing, MN': 1212903, 'Watertown, NY': 1036102, 'Sitka, AK': 1482802, 'Chico, CA': 1100202, 'Laredo, TX': 1306104, 'Aberdeen, SD': 1014102, 'Latrobe, PA': 1289803, 'Christiansted, VI': 1502704, 'Jamestown, ND': 1251902, 'Ponce, PR': 1425403, 'Juneau, AK': 1252304, 'Beaumont/Port Arthur, TX': 1072804, 'New Bern/Morehead/Beaufort, NC': 1161706, 'Bangor, ME': 1058102, 'Grand Island, NE': 1198002, 'Alpena, MI': 1033302, 'Rhinelander, WI': 1452002, 'Vernal, UT': 1558202, 'Columbus, MS': 1200702, 'Champaign/Urbana, IL': 1106702, 'Nome, AK': 1387303, 'Bellingham, WA': 1066602, 'Lawton/Fort Sill, OK': 1289102, 'Hays, KS': 1225502, 'St. Cloud, MN': 1500802, 'Islip, NY': 1239102, 'Santa Maria, CA': 1490503, 'Bethel, AK': 1055102, 'Dillingham, AK': 1133602, 'Lewiston, ID': 1312702, 'Topeka, KS': 1172602, 'Gunnison, CO': 1201203, 'Marquette, MI': 1345902, 'Klamath Falls, OR': 1302402, 'Deadhorse, AK': 1470903, 'Ketchikan, AK': 1281902, 'Petersburg, AK': 1425603, 'Waterloo, IA': 1026802, 'Longview, TX': 1190502, 'Barrow, AK': 1075402, 'Bemidji, MN': 1063104, 'Yakutat, AK': 1599102, 'Cedar City, UT': 1091802, 'Eagle, CO': 1150303, 'Iron Mountain/Kingsfd, MI': 1233502, 'Toledo, OH': 1529502, 'Escanaba, MI': 1158702, 'Guam, TT': 1201602, 'Niagara Falls, NY': 1226503, 'Laramie, WY': 1288802, 'Cody, WY': 1109702, 'Moab, UT': 1109202, 'Gustavus, AK': 1199702, 'Roswell, NM': 1458801, 'Cordova, AK': 1092603, 'Wrangell, AK': 1584102, 'King Salmon, AK': 1024502, 'Hyannis, MA': 1225002, "Martha's Vineyard, MA": 1354102, 'Kodiak, AK': 1017001, 'West Yellowstone, MT': 1589702, 'Saipan, TT': 1495503, 'Macon, GA': 1320302}

MODEL_PATH = "/home/jennifer/Documentos/tercer_año/segundo_semestre/ID/AeroData/app/predictions/flight_delay_model.cbm"

# =============== 1. Cargar datos de vuelos ==================
def fetch_flight_data():
    url = f'https://api.aviationstack.com/v1/flights?access_key={API_KEY}&flight_status=landed'
    response = requests.get(url)
    response.raise_for_status()
    raw_data = response.json()
    if not raw_data.get('data'):
        raise ValueError("No se encontraron datos de vuelos")
    return pd.DataFrame(raw_data['data'])


def extract_flight_fields(flights_df):
    flights_df['op_carrier'] = flights_df['airline'].apply(lambda x: x.get('iata') if isinstance(x, dict) else None)
    flights_df['op_carrier_fl_num'] = flights_df['flight'].apply(lambda x: x.get('number') if isinstance(x, dict) else None)

    flights_df['origin_airport_id'] = flights_df['departure'].apply(lambda x: x.get('iata') if isinstance(x, dict) else None)
    flights_df['dest_airport_id'] = flights_df['arrival'].apply(lambda x: x.get('iata') if isinstance(x, dict) else None)

    # Tiempos programados y reales
    flights_df['scheduled_dep_time'] = pd.to_datetime(flights_df['departure'].apply(lambda x: x.get('scheduled'))).dt.strftime('%H:%M:%S')
    flights_df['scheduled_arr_time'] = pd.to_datetime(flights_df['arrival'].apply(lambda x: x.get('scheduled'))).dt.strftime('%H:%M:%S')

    flights_df['actual_dep_time'] = pd.to_datetime(flights_df['departure'].apply(lambda x: x.get('actual')))
    flights_df['actual_arr_time'] = pd.to_datetime(flights_df['arrival'].apply(lambda x: x.get('actual')))

    flights_df['actual_dep_time'] = flights_df['actual_dep_time'].dt.strftime('%H:%M:%S')
    flights_df['actual_arr_time'] = flights_df['actual_arr_time'].dt.strftime('%H:%M:%S')

    return flights_df


# ================ 2. Cargar aeropuertos y coordenadas ==================
def fetch_all_airport_coords(api_key):
    base_url = f'https://api.aviationstack.com/v1/airports?access_key={api_key}'
    airport_coords = {}
    for offset in range(0, 10000, 1000):
        url = f'{base_url}&limit=1000&offset={offset}'
        response = requests.get(url)
        data = response.json()
        if data.get('data'):
            for airport in data['data']:
                iata = airport.get('iata_code')
                lat = airport.get('latitude')
                lon = airport.get('longitude')
                if iata and lat and lon:
                    airport_coords[iata] = (float(lat), float(lon))
        else:
            break
    print(f"✅ Coordenadas cargadas: {len(airport_coords)} aeropuertos")
    return airport_coords


# ================ 3. Calcular distancia ==================
def calculate_distance(row, airport_coords):
    origin_coords = airport_coords.get(row['origin_airport_id'])
    dest_coords = airport_coords.get(row['dest_airport_id'])
    if origin_coords and dest_coords:
        return geodesic(origin_coords, dest_coords).miles
    return None


# ================ 4. Validar retraso ==================
def calculate_delay(row):
    try:
        scheduled = datetime.strptime(row['scheduled_arr_time'], '%H:%M:%S')
        actual = datetime.strptime(row['actual_arr_time'], '%H:%M:%S')
        delay_minutes = (actual - scheduled).total_seconds() / 60
        return delay_minutes
    except Exception:
        return None

def get_or_create_airport_id(iata_code):
    global current_id
    if pd.isna(iata_code) or iata_code not in airport_coords:
        return None
    
    if iata_code in iata_to_id_dict:
        return iata_to_id_dict[iata_code]
    else:
        current_id += 1
        iata_to_id_dict[iata_code] = str(current_id)
        return str(current_id)
    
def calculate_duration(row):
    dep_time = datetime.strptime(row['scheduled_dep_time'], '%H:%M:%S').time()
    arr_time = datetime.strptime(row['scheduled_arr_time'], '%H:%M:%S').time()

    dep_datetime = datetime.combine(datetime.min, dep_time)
    arr_datetime = datetime.combine(datetime.min, arr_time)

    if arr_datetime < dep_datetime:
        arr_datetime += timedelta(days=1)  # Para vuelos que cruzan la medianoche
    return (arr_datetime - dep_datetime).total_seconds() / 60

# ================ 5. Preparar y predecir ==================
def prepare_for_validation(flight_df, predictions):
    flight_df = flight_df.copy()
    flight_df['predicted_delay'] = predictions
    flight_df['actual_delay'] = flight_df.apply(calculate_delay, axis=1)
    return flight_df


# ================= MAIN ==================
flights_df = fetch_flight_data()
flights_df = extract_flight_fields(flights_df)

airport_coords = fetch_all_airport_coords(API_KEY)

flights_df['distance'] = flights_df.apply(lambda row: calculate_distance(row, airport_coords), axis=1)

# Aquí puedes añadir más features según tu modelo
flights_df['day_of_week'] = datetime.today().weekday()
flights_df['month'] = datetime.today().month
flights_df['is_weekend'] = flights_df['day_of_week'].isin([5,6]).astype(int)

# dummy para op_unique_carrier (ajusta si necesario)
flights_df['op_unique_carrier'] = flights_df['op_carrier']
flights_df['scheduled_duration'] = flights_df.apply(calculate_duration, axis=1)
flights_df['dep_hour'] = flights_df['scheduled_dep_time'].apply(lambda x: int(x.split(':')[0]) if pd.notnull(x) else None)
flights_df['dep_minute'] = flights_df['scheduled_dep_time'].apply(lambda x: int(x.split(':')[1]) if pd.notnull(x) else None)
flights_df['arr_hour'] = flights_df['scheduled_arr_time'].apply(lambda x: int(x.split(':')[0]) if pd.notnull(x) else None)


current_id = 1039705

flights_df['origin_airport_id'] = flights_df['origin_airport_id'].apply(get_or_create_airport_id)
flights_df['dest_airport_id'] = flights_df['dest_airport_id'].apply(get_or_create_airport_id)

model_features = [
    "op_unique_carrier","op_carrier","origin_airport_id","dest_airport_id",
    "distance","op_carrier_fl_num","day_of_week","month","is_weekend",
    "scheduled_duration","dep_hour","dep_minute","arr_hour"
]

flights_df = flights_df.dropna(subset=model_features)
features = flights_df[model_features]
# ================ PREDICCIÓN ==================
model = CatBoostRegressor()
model.load_model(MODEL_PATH)

prediction_pool = Pool(
    data=features,
    cat_features=[
        "op_unique_carrier","op_carrier","op_carrier_fl_num",
        "origin_airport_id","dest_airport_id","day_of_week","month","dep_hour","arr_hour"
    ]
)

predictions = model.predict(prediction_pool)

# VALIDACIÓN
validation_data = prepare_for_validation(flights_df, predictions)

print(validation_data[[
    'op_carrier', 'op_carrier_fl_num', 'origin_airport_id', 'dest_airport_id',
    'scheduled_arr_time', 'actual_arr_time', 'actual_delay', 'predicted_delay'
]].head())


✅ Coordenadas cargadas: 6705 aeropuertos
  op_carrier op_carrier_fl_num origin_airport_id dest_airport_id  \
0         QF              7420           1039706         1039727   
1         MH               132           1039707         1039714   
2         PR               221           1039708         1039727   
3         CF              9017           1039709         1039728   
4         FJ               252           1039710         1039729   

  scheduled_arr_time actual_arr_time  actual_delay  predicted_delay  
0           01:35:00        01:56:00          21.0         2.978398  
1           07:55:00        07:39:00         -16.0        -3.576542  
2           09:55:00        09:54:00          -1.0       -13.338097  
3           02:50:00        02:52:00           2.0       -12.886337  
4           06:05:00        05:52:00         -13.0        10.759222  


In [55]:
from sqlalchemy import create_engine, MetaData, Table
import os
import pandas as pd
from datetime import datetime,timedelta
from dotenv import load_dotenv
import requests
from geopy.distance import geodesic

load_dotenv()

API_KEY=os.getenv('API_KEY')
API_KEY="0758dc493f393a5024a043190590116f"
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")  # Mejor usa una variable para la contraseña
db_host = os.getenv("DB_HOST", "localhost")
db_port = os.getenv("DB_PORT", "5432")  # Puerto por defecto de PostgreSQL
db_name = os.getenv("DB_NAME")

db_url = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}?sslmode=require"

# Crear motor
engine = create_engine(db_url)
def calc_duration(row):
    try:
        dep = datetime.strptime(row['scheduled_dep_time'], '%H:%M:%S')
        arr = datetime.strptime(row['scheduled_arr_time'], '%H:%M:%S')
        if arr < dep:
            arr += timedelta(days=1)  # cruza medianoche
        return (arr - dep).total_seconds() / 60
    except Exception:
        return None


def calc_distance(row, airport_coords):
    origin = airport_coords.get(row['origin_airport_id'])
    dest = airport_coords.get(row['dest_airport_id'])
    if origin and dest:
        return geodesic(origin, dest).miles
    return None

def fetch_all_airport_coords(api_key):
    base_url = f'https://api.aviationstack.com/v1/airports?access_key={api_key}'
    airport_coords = {}
    for offset in range(0, 10000, 1000):
        url = f'{base_url}&limit=1000&offset={offset}'
        response = requests.get(url)
        data = response.json()
        if data.get('data'):
            for airport in data['data']:
                iata = airport.get('iata_code')
                lat = airport.get('latitude')
                lon = airport.get('longitude')
                if iata and lat and lon:
                    airport_coords[iata] = (float(lat), float(lon))
        else:
            break
    print(f"✅ Coordenadas cargadas: {len(airport_coords)} aeropuertos")
    return airport_coords

def check_today_data():
    today_str = datetime.now().date().isoformat()
    res = engine.table("vuelos").select("*").eq("date", today_str).execute()
    data = res.data
    if data:
        return pd.DataFrame(data)
    return None

def save_today_data(df):
    df["date"] = datetime.now().date().isoformat()
    engine.table("vuelos").insert(df.to_dict(orient="records")).execute()

# def fetch_flights_from_api():

#     url = f"https://api.aviationstack.com/v1/flights?access_key={API_KEY}&flight_status=scheduled"
#     response = requests.get(url)
#     response.raise_for_status()
#     raw_data = response.json()
#     if not raw_data.get("data"):
#         raise ValueError("No se encontraron vuelos programados")
#     df = pd.DataFrame(raw_data["data"])
#     return df
def fetch_flights_from_api(hours_ahead=24, max_results=1000):
    base_url = "https://api.aviationstack.com/v1/flights"
    all_flights = []
    
    # Configuración básica de parámetros
    params = {
        'access_key': API_KEY,
        'flight_status': 'scheduled',
        'limit': 100  # Máximo permitido por request
    }
    
    # Si quieres vuelos para todo el día, no uses filtros temporales
    # Si prefieres un rango específico:
    if hours_ahead:
        now = datetime.utcnow()
        time_limit = now + timedelta(hours=hours_ahead)
        params.update({
            'dep_estimated_from': now.strftime('%Y-%m-%dT%H:%M:%S'),
            'dep_estimated_to': time_limit.strftime('%Y-%m-%dT%H:%M:%S')
        })
    
    # Implementación de paginación
    offset = 0
    while len(all_flights) < max_results:
        params['offset'] = offset
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        raw_data = response.json()
        
        if not raw_data.get("data"):
            break
            
        all_flights.extend(raw_data["data"])
        offset += len(raw_data["data"])
        
        # Si obtenemos menos resultados que el límite, es el final
        if len(raw_data["data"]) < params['limit']:
            break
    
    return pd.DataFrame(all_flights)
def get_flights():
    df = check_today_data()
    if df is not None:
        print("✅ Datos cargados de Supabase")
        return df

    print("📡 No hay datos en Supabase, consultando API…")
    df = fetch_flights_from_api()
    df = transform_df(df)  # transformar columnas como necesites
    save_today_data(df)
    return df

def transform_df(df):
    # Extraer datos básicos
    df['op_carrier'] = df['airline'].apply(lambda x: x.get('iata') if isinstance(x, dict) else None)
    df['op_carrier_fl_num'] = df['flight'].apply(lambda x: x.get('number') if isinstance(x, dict) else None)
    
    # Aeropuertos de origen y destino
    df['origin_airport_id'] = df['departure'].apply(lambda x: x.get('iata') if isinstance(x, dict) else None)
    df['dest_airport_id'] = df['arrival'].apply(lambda x: x.get('iata') if isinstance(x, dict) else None)
    
    # NOMBRES DE LOS AEROPUERTOS (lo que quieres)
    df['departure_airport_name'] = df['departure'].apply(lambda x: x.get('airport') if isinstance(x, dict) else None)
    df['arrival_airport_name'] = df['arrival'].apply(lambda x: x.get('airport') if isinstance(x, dict) else None)
    
    df['airline_name'] = df['airline'].apply(lambda x: x.get('name') if isinstance(x, dict) else None)

    # Fechas programadas
    df['scheduled_dep'] = pd.to_datetime(
        df['departure'].apply(lambda x: x.get('scheduled') if isinstance(x, dict) else None),
        errors='coerce'
    ).dt.strftime("%Y-%m-%d %H:%M:%S")
    df['scheduled_arr'] = pd.to_datetime(
        df['arrival'].apply(lambda x: x.get('scheduled') if isinstance(x, dict) else None),
        errors='coerce'
    ).dt.strftime("%Y-%m-%d %H:%M:%S")
    # Verificar si hay horas mayores a 12 (formato 24h)

    df['scheduled_dep_time'] = pd.to_datetime(df['scheduled_dep']).dt.strftime('%H:%M:%S')
    df['scheduled_arr_time'] = pd.to_datetime(df['scheduled_arr']).dt.strftime('%H:%M:%S')


    df['dep_hour'] = df['scheduled_dep_time'].apply(lambda x: int(x.split(':')[0]) if pd.notnull(x) else None)
    df['dep_minute'] = df['scheduled_dep_time'].apply(lambda x: int(x.split(':')[1]) if pd.notnull(x) else None)
    df['arr_hour'] = df['scheduled_arr_time'].apply(lambda x: int(x.split(':')[0]) if pd.notnull(x) else None)

    df['day_of_week'] = datetime.today().weekday()
    df['month'] = datetime.today().month
    df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)

    df['scheduled_duration'] = df.apply(calc_duration, axis=1)

    airport_coords = fetch_all_airport_coords(API_KEY)
    df['distance'] = df.apply(lambda row: calc_distance(row, airport_coords), axis=1)

    # Eliminar columnas anidadas si ya no se necesitan
    df.drop(columns=['departure', 'arrival',"live","aircraft","airline","flight"], inplace=True, errors='ignore')

    return df

df = fetch_flights_from_api()
df = transform_df(df) 
print(df.head(1))

/tmp/ipykernel_10450/2854700119.py:96: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()


✅ Coordenadas cargadas: 6705 aeropuertos
  flight_date flight_status op_carrier op_carrier_fl_num origin_airport_id  \
0  2025-07-10     scheduled         QF              7420               SYD   

  dest_airport_id          departure_airport_name    arrival_airport_name  \
0             BNE  Sydney Kingsford Smith Airport  Brisbane International   

  airline_name        scheduled_dep  ... scheduled_dep_time  \
0       Qantas  2025-07-10 00:05:00  ...           00:05:00   

  scheduled_arr_time dep_hour  dep_minute  arr_hour  day_of_week  month  \
0           01:35:00        0           5         1            2      7   

   is_weekend  scheduled_duration    distance  
0           0                90.0  464.235681  

[1 rows x 21 columns]


In [57]:
print(len(df))


1000


In [50]:
print(set(df["scheduled_arr_time"]))

{'03:05:00', '01:20:00', '04:10:00', '02:25:00', '06:30:00', '05:40:00', '04:55:00', '04:05:00', '02:20:00', '02:10:00', '15:05:00', '02:35:00', '02:05:00', '04:20:00', '07:35:00', '07:05:00', '03:10:00', '03:20:00', '08:15:00', '08:45:00', '05:30:00', '01:55:00', '06:00:00', '02:00:00', '08:05:00', '04:40:00', '03:45:00', '05:00:00', '01:40:00', '04:35:00', '06:20:00', '03:30:00', '07:15:00', '05:50:00', '06:05:00', '09:00:00', '05:10:00', '04:25:00', '06:50:00', '07:10:00', '05:25:00', '03:15:00', '04:00:00', '01:30:00', '07:40:00', '04:45:00', '01:35:00'}


In [61]:
print(sorted(set(df["scheduled_dep_time"])))
print(sorted(set(df["scheduled_arr_time"])))
print(set(df["distance"]))

['00:00:00', '00:01:00', '00:05:00', '00:10:00', '00:15:00', '00:20:00', '00:25:00', '00:30:00', '00:35:00', '00:40:00', '00:45:00', '00:50:00', '00:55:00', '01:00:00', '01:05:00', '01:10:00', '01:15:00', '01:20:00', '01:25:00', '01:30:00', '01:35:00', '01:40:00', '01:45:00', '01:50:00', '01:55:00', '02:00:00', '02:05:00', '02:10:00', '02:15:00', '02:20:00', '02:25:00', '02:30:00', '02:35:00', '02:39:00', '02:40:00', '02:45:00', '02:50:00', '02:55:00', '03:00:00', '03:05:00', '03:10:00', '03:15:00', '03:20:00', '03:25:00', '03:30:00', '03:35:00', '03:40:00', '03:45:00', '03:50:00', '03:55:00', '04:00:00', '04:05:00', '04:10:00', '04:15:00', '04:20:00', '04:25:00', '04:30:00', '04:35:00', '04:40:00', '04:45:00', '04:50:00', '04:55:00', '05:05:00', '05:10:00', '05:15:00', '05:20:00', '05:25:00', '05:35:00', '05:40:00', '05:45:00', '05:50:00', '05:55:00', '06:00:00', '06:01:00', '06:05:00', '06:10:00', '06:15:00', '06:20:00', '06:25:00', '06:30:00', '06:35:00', '06:40:00', '06:45:00', '06

In [62]:
df.columns

Index(['flight_date', 'flight_status', 'op_carrier', 'op_carrier_fl_num',
       'origin_airport_id', 'dest_airport_id', 'departure_airport_name',
       'arrival_airport_name', 'airline_name', 'scheduled_dep',
       'scheduled_arr', 'scheduled_dep_time', 'scheduled_arr_time', 'dep_hour',
       'dep_minute', 'arr_hour', 'day_of_week', 'month', 'is_weekend',
       'scheduled_duration', 'distance'],
      dtype='object')

In [ ]:
datos=[(10599, 1059904, '30599', 'BHM', 'Birmingham, AL', 'AL', '01', 'Alabama', 51),
(12953, 1295304, '31703', 'LGA', 'New York, NY', 'NY', '36', 'New York', 22),
(10397, 1039707, '30397', 'ATL', 'Atlanta, GA', 'GA', '13', 'Georgia', 34),
(13422, 1342202, '30562', 'MOB', 'Mobile, AL', 'AL', '01', 'Alabama', 51),
(12217, 1221702, '30255', 'HSV', 'Huntsville, AL', 'AL', '01', 'Alabama', 51),
(12266, 1226603, '31453', 'IAH', 'Houston, TX', 'TX', '48', 'Texas', 74),
(11298, 1129806, '30194', 'DFW', 'Dallas/Fort Worth, TX', 'TX', '48', 'Texas', 74),
(13930, 1393007, '30977', 'ORD', 'Chicago, IL', 'IL', '17', 'Illinois', 41),
(13277, 1327702, '33277', 'MGM', 'Montgomery, AL', 'AL', '01', 'Alabama', 51),
(13303, 1330303, '32467', 'MIA', 'Miami, FL', 'FL', '12', 'Florida', 33),
(11278, 1127805, '30852', 'DCA', 'Washington, DC', 'VA', '51', 'Virginia', 38),
(11057, 1105703, '31057', 'CLT', 'Charlotte, NC', 'NC', '37', 'North Carolina', 36),
(14100, 1410005, '34100', 'PHL', 'Philadelphia, PA', 'PA', '42', 'Pennsylvania', 23),
(11292, 1129202, '30325', 'DEN', 'Denver, CO', 'CO', '08', 'Colorado', 82),
(11308, 1130802, '31308', 'DHN', 'Dothan, AL', 'AL', '01', 'Alabama', 51),
(11433, 1143302, '31295', 'DTW', 'Detroit, MI', 'MI', '26', 'Michigan', 43),
(13232, 1323202, '30977', 'MDW', 'Chicago, IL', 'IL', '17', 'Illinois', 41),
(11697, 1169706, '32467', 'FLL', 'Fort Lauderdale, FL', 'FL', '12', 'Florida', 33),
(11259, 1125904, '30194', 'DAL', 'Dallas, TX', 'TX', '48', 'Texas', 74),
(15304, 1530402, '33195', 'TPA', 'Tampa, FL', 'FL', '12', 'Florida', 33),
(13204, 1320402, '31454', 'MCO', 'Orlando, FL', 'FL', '12', 'Florida', 33),
(10821, 1082106, '30852', 'BWI', 'Baltimore, MD', 'MD', '24', 'Maryland', 35),
(12191, 1219102, '31453', 'HOU', 'Houston, TX', 'TX', '48', 'Texas', 74),
(12889, 1288903, '32211', 'LAS', 'Las Vegas, NV', 'NV', '32', 'Nevada', 85),
(12264, 1226402, '30852', 'IAD', 'Washington, DC', 'VA', '51', 'Virginia', 38),
(10562, 1056202, '30562', 'BFM', 'Mobile, AL', 'AL', '01', 'Alabama', 51),
(13930, 1393008, '30977', 'ORD', 'Chicago, IL', 'IL', '17', 'Illinois', 41),
(12191, 1219103, '31453', 'HOU', 'Houston, TX', 'TX', '48', 'Texas', 74),
(15919, 1591905, '31834', 'XNA', 'Fayetteville, AR', 'AR', '05', 'Arkansas', 71),
(12889, 1288904, '32211', 'LAS', 'Las Vegas, NV', 'NV', '32', 'Nevada', 85),
(13487, 1348702, '31650', 'MSP', 'Minneapolis, MN', 'MN', '27', 'Minnesota', 63),
(11049, 1104903, '31049', 'CLL', 'College Station/Bryan, TX', 'TX', '48', 'Texas', 74),
(14107, 1410702, '30466', 'PHX', 'Phoenix, AZ', 'AZ', '04', 'Arizona', 81),
(10693, 1069302, '30693', 'BNA', 'Nashville, TN', 'TN', '47', 'Tennessee', 54),
(13851, 1385103, '33851', 'OKC', 'Oklahoma City, OK', 'OK', '40', 'Oklahoma', 73),
(10781, 1078105, '30781', 'BTR', 'Baton Rouge, LA', 'LA', '22', 'Louisiana', 72),
(12339, 1233904, '32337', 'IND', 'Indianapolis, IN', 'IN', '18', 'Indiana', 42),
(13198, 1319801, '33198', 'MCI', 'Kansas City, MO', 'MO', '29', 'Missouri', 64),
(10423, 1042302, '30423', 'AUS', 'Austin, TX', 'TX', '48', 'Texas', 74),
(13930, 1393006, '30977', 'ORD', 'Chicago, IL', 'IL', '17', 'Illinois', 41),
(11259, 1125903, '30194', 'DAL', 'Dallas, TX', 'TX', '48', 'Texas', 74),
(10868, 1086803, '30868', 'CAE', 'Columbia, SC', 'SC', '45', 'South Carolina', 37),
(13485, 1348502, '33485', 'MSN', 'Madison, WI', 'WI', '55', 'Wisconsin', 45),
(13342, 1334207, '33342', 'MKE', 'Milwaukee, WI', 'WI', '55', 'Wisconsin', 45),
(12892, 1289208, '32575', 'LAX', 'Los Angeles, CA', 'CA', '06', 'California', 91),
(15919, 1591904, '31834', 'XNA', 'Fayetteville, AR', 'AR', '05', 'Arkansas', 71),
(1039705, 30397, 'ATL', 'Atlanta, GA', 'GA', '13', 'Georgia', '34', 14057),
(1379602, 32457, 'OAK', 'Oakland, CA', 'CA', '6', 'California', '91', 14869),
(1143302, 31295, 'DTW', 'Detroit, MI', 'MI', '26', 'Michigan', '43', 15304),
(1319801, 33198, 'MCI', 'Kansas City, MO', 'MO', '29', 'Missouri', '64', 13487),
(1288903, 32211, 'LAS', 'Las Vegas, NV', 'NV', '32', 'Nevada', '85', 14869),
(1477101, 32457, 'SFO', 'San Francisco, CA', 'CA', '6', 'California', '91', 14869),
(1247802, 31703, 'JFK', 'New York, NY', 'NY', '36', 'New York', '22', 15304),
(1348702, 31650, 'MSP', 'Minneapolis, MN', 'MN', '27', 'Minnesota', '63', 10721),
(1289203, 32575, 'LAX', 'Los Angeles, CA', 'CA', '6', 'California', '91', 10693),
(1161802, 31703, 'EWR', 'Newark, NJ', 'NJ', '34', 'New Jersey', '21', 13230),
(1334205, 33342, 'MKE', 'Milwaukee, WI', 'WI', '55', 'Wisconsin', '45', 12266),
(1393003, 30977, 'ORD', 'Chicago, IL', 'IL', '17', 'Illinois', '41', 11986),
(1127802, 30852, 'DCA', 'Washington, DC', 'VA', '51', 'Virginia', '38', 11618),
(1226402, 30852, 'IAD', 'Washington, DC', 'VA', '51', 'Virginia', '38', 11995),
(1082103, 30852, 'BWI', 'Baltimore, MD', 'MD', '24', 'Maryland', '35', 10721),
(1320402, 31454, 'MCO', 'Orlando, FL', 'FL', '12', 'Florida', '33', 10792),
(1104202, 30647, 'CLE', 'Cleveland, OH', 'OH', '39', 'Ohio', '44', 14307),
(1501603, 31123, 'STL', 'St. Louis, MO', 'MO', '29', 'Missouri', '64', 11292),
(1115003, 31150, 'CSG', 'Columbus, GA', 'GA', '13', 'Georgia', '34', 10397),
(1182304, 31823, 'FWA', 'Fort Wayne, IN', 'IN', '18', 'Indiana', '42', 10397),
(1197702, 31977, 'GRB', 'Green Bay, WI', 'WI', '55', 'Wisconsin', '45', 11433),
(1232002, 32320, 'ILG', 'Wilmington, DE', 'DE', '10', 'Delaware', '31', 13232),
(1410702, 30466, 'PHX', 'Phoenix, AZ', 'AZ', '4', 'Arizona', '81', 12173),
(1129803, 30194, 'DFW', 'Dallas/Fort Worth, TX', 'TX', '48', 'Texas', '74', 12915),
(1105703, 31057, 'CLT', 'Charlotte, NC', 'NC', '37', 'North Carolina', '36', 12889),
(1410002, 34100, 'PHL', 'Philadelphia, PA', 'PA', '42', 'Pennsylvania', '23', 12266),
(1129202, 30325, 'DEN', 'Denver, CO', 'CO', '8', 'Colorado', '82', 12264),
(1412202, 30198, 'PIT', 'Pittsburgh, PA', 'PA', '42', 'Pennsylvania', '23', 10821),
(1449202, 34492, 'RDU', 'Raleigh/Durham, NC', 'NC', '37', 'North Carolina', '36', 10693),
(1014002, 30140, 'ABQ', 'Albuquerque, NM', 'NM', '35', 'New Mexico', '86', 13796),
(1099402, 30994, 'CHS', 'Charleston, SC', 'SC', '45', 'South Carolina', '37', 10693),
(1126702, 31267, 'DAY', 'Dayton, OH', 'OH', '39', 'Ohio', '44', 11292),
(1537002, 34653, 'TUL', 'Tulsa, OK', 'OK', '40', 'Oklahoma', '73', 11259),
(1245102, 31136, 'JAX', 'Jacksonville, FL', 'FL', '12', 'Florida', '33', 13232),
(1323202, 30977, 'MDW', 'Chicago, IL', 'IL', '17', 'Illinois', '41', 11042),
(1468303, 33214, 'SAT', 'San Antonio, TX', 'TX', '48', 'Texas', '74', 13232),
(1056103, 30561, 'BFL', 'Bakersfield, CA', 'CA', '6', 'California', '91', 14107),
(1486903, 34614, 'SLC', 'Salt Lake City, UT', 'UT', '49', 'Utah', '87', 13851),
(1405702, 34057, 'PDX', 'Portland, OR', 'OR', '41', 'Oregon', '92', 11603),
(1217301, 32134, 'HNL', 'Honolulu, HI', 'HI', '15', 'Hawaii', '2', 12264),
(1226603, 31453, 'IAH', 'Houston, TX', 'TX', '48', 'Texas', '74', 12478),
(1330303, 32467, 'MIA', 'Miami, FL', 'FL', '12', 'Florida', '33', 11298),
(1052904, 30529, 'BDL', 'Hartford, CT', 'CT', '9', 'Connecticut', '11', 12892),
(1217302, 32134, 'HNL', 'Honolulu, HI', 'HI', '15', 'Hawaii', '2', 14679),
(1014003, 30140, 'ABQ', 'Albuquerque, NM', 'NM', '35', 'New Mexico', '86', 11259),
(1104203, 30647, 'CLE', 'Cleveland, OH', 'OH', '39', 'Ohio', '44', 11697),
(1114604, 31146, 'CRW', 'Charleston/Dunbar, WV', 'WV', '54', 'West Virginia', '39', 11433),
(1299204, 32600, 'LIT', 'Little Rock, AR', 'AR', '5', 'Arkansas', '71', 12953),
(1338801, 33388, 'MMH', 'Mammoth Lakes, CA', 'CA', '6', 'California', '91', 11292),
(1294503, 32945, 'LEX', 'Lexington, KY', 'KY', '21', 'Kentucky', '52', 11278),
(1502403, 34945, 'STT', 'Charlotte Amalie, VI', 'VI', '78', 'U.S. Virgin Islands', '4', 11697),
(1015804, 30158, 'ACY', 'Atlantic City, NJ', 'NJ', '34', 'New Jersey', '21', 15304),
(1169703, 32467, 'FLL', 'Fort Lauderdale, FL', 'FL', '12', 'Florida', '33', 10732),
(1127803, 30852, 'DCA', 'Washington, DC', 'VA', '51', 'Virginia', '38', 13495),
(1393004, 30977, 'ORD', 'Chicago, IL', 'IL', '17', 'Illinois', '41', 10693),
(1169704, 32467, 'FLL', 'Fort Lauderdale, FL', 'FL', '12', 'Florida', '33', 13577),
(1379604, 32457, 'OAK', 'Oakland, CA', 'CA', '6', 'California', '91', 13930),
(1477102, 32457, 'SFO', 'San Francisco, CA', 'CA', '6', 'California', '91', 10561),
(1396403, 33964, 'OTH', 'North Bend/Coos Bay, OR', 'OR', '41', 'Oregon', '92', 11292),
(1142304, 31423, 'DSM', 'Des Moines, IA', 'IA', '19', 'Iowa', '61', 10397),
(1244805, 32448, 'JAN', 'Jackson/Vicksburg, MS', 'MS', '28', 'Mississippi', '53', 14108),
(1357702, 31135, 'MYR', 'Myrtle Beach, SC', 'SC', '45', 'South Carolina', '37', 11146),
(1163002, 31517, 'FAI', 'Fairbanks, AK', 'AK', '2', 'Alaska', '1', 1955),
(1474703, 30559, 'SEA', 'Seattle, WA', 'WA', '53', 'Washington', '93', 1445),
(1119302, 33105, 'CVG', 'Cincinnati, OH', 'KY', '21', 'Kentucky', '52', 1700),
(1530402, 33195, 'TPA', 'Tampa, FL', 'FL', '12', 'Florida', '33', 1615),
(1498603, 34986, 'SRQ', 'Sarasota/Bradenton, FL', 'FL', '12', 'Florida', '33', 1930),
(1385103, 33851, 'OKC', 'Oklahoma City, OK', 'OK', '40', 'Oklahoma', '73', 805),
(1291503, 31205, 'LCH', 'Lake Charles, LA', 'LA', '22', 'Louisiana', '72', 752),
(1114002, 31140, 'CRP', 'Corpus Christi, TX', 'TX', '48', 'Texas', '74', 1816),
(1393102, 33667, 'ORF', 'Norfolk, VA', 'VA', '51', 'Virginia', '38', 1230),
(1295302, 31703, 'LGA', 'New York, NY', 'NY', '36', 'New York', '22', 1819),
(1387102, 33316, 'OMA', 'Omaha, NE', 'NE', '31', 'Nebraska', '65', 1625),
(1059904, 30599, 'BHM', 'Birmingham, AL', 'AL', '1', 'Alabama', '51', 1246),
(1198603, 31986, 'GRR', 'Grand Rapids, MI', 'MI', '26', 'Michigan', '43', 858),
(1106603, 31066, 'CMH', 'Columbus, OH', 'OH', '39', 'Ohio', '44', 1018),
(1509602, 35096, 'SYR', 'Syracuse, NY', 'NY', '36', 'New York', '22', 1124),
(1100303, 31003, 'CID', 'Cedar Rapids/Iowa City, IA', 'IA', '19', 'Iowa', '61', 2133),
(1227802, 30928, 'ICT', 'Wichita, KS', 'KS', '20', 'Kansas', '62', 820),
(1430702, 30721, 'PVD', 'Providence, RI', 'RI', '44', 'Rhode Island', '15', 1247),
(1463502, 31714, 'RSW', 'Fort Myers, FL', 'FL', '12', 'Florida', '33', 1612),
(1072102, 30721, 'BOS', 'Boston, MA', 'MA', '25', 'Massachusetts', '13', 707),
(1402702, 34027, 'PBI', 'West Palm Beach/Palm Beach, FL', 'FL', '12', 'Florida', '33', 1640),
(1086803, 30868, 'CAE', 'Columbia, SC', 'SC', '45', 'South Carolina', '37', 1835),
(1043102, 30431, 'AVL', 'Asheville, NC', 'NC', '37', 'North Carolina', '36', 1324),
(1233904, 32337, 'IND', 'Indianapolis, IN', 'IN', '18', 'Indiana', '42', 722),
(1468502, 34685, 'SAV', 'Savannah, GA', 'GA', '13', 'Georgia', '34', 1935),
(1199603, 31871, 'GSP', 'Greer, SC', 'SC', '45', 'South Carolina', '37', 600),
(1125903, 30194, 'DAL', 'Dallas, TX', 'TX', '48', 'Texas', '74', 1052),
(1221702, 30255, 'HSV', 'Huntsville, AL', 'AL', '1', 'Alabama', '51', 1007),
(1452401, 34524, 'RIC', 'Richmond, VA', 'VA', '51', 'Virginia', '38', 1304),
(1324402, 33244, 'MEM', 'Memphis, TN', 'TN', '47', 'Tennessee', '54', 1958),
(1409803, 33667, 'PHF', 'Newport News/Williamsburg, VA', 'VA', '51', 'Virginia', '38', 2030),
(1177502, 31775, 'FSD', 'Sioux Falls, SD', 'SD', '46', 'South Dakota', '67', 2140),
(1323002, 32070, 'MDT', 'Harrisburg, PA', 'PA', '42', 'Pennsylvania', '23', 2153),
(1219102, 31453, 'HOU', 'Houston, TX', 'TX', '48', 'Texas', '74', 1041),
(1295103, 32951, 'LFT', 'Lafayette, LA', 'LA', '22', 'Louisiana', '72', 2006),
(1481402, 30476, 'SHV', 'Shreveport, LA', 'LA', '22', 'Louisiana', '72', 1730),
(1327702, 33277, 'MGM', 'Montgomery, AL', 'AL', '1', 'Alabama', '51', 2040),
(1329604, 30721, 'MHT', 'Manchester, NH', 'NH', '33', 'New Hampshire', '14', 1015),
(1478302, 34783, 'SGF', 'Springfield, MO', 'MO', '29', 'Missouri', '64', 2005),
(1541202, 35412, 'TYS', 'Knoxville, TN', 'TN', '47', 'Tennessee', '54', 1535),
(1020803, 30208, 'AGS', 'Augusta, GA', 'GA', '13', 'Georgia', '34', 840),
(1079204, 30792, 'BUF', 'Buffalo, NY', 'NY', '36', 'New York', '22', 1453),
(1410803, 34108, 'PIA', 'Peoria, IL', 'IL', '17', 'Illinois', '41', 945),
(1199502, 31995, 'GSO', 'Greensboro/High Point, NC', 'NC', '37', 'North Carolina', '36', 1648),
(1350202, 33502, 'MTJ', 'Montrose/Delta, CO', 'CO', '8', 'Colorado', '82', 1604),
(1069302, 30693, 'BNA', 'Nashville, TN', 'TN', '47', 'Tennessee', '54', 1712),
(1141303, 30285, 'DRO', 'Durango, CO', 'CO', '8', 'Colorado', '82', 806),
(1027903, 30279, 'AMA', 'Amarillo, TX', 'TX', '48', 'Texas', '74', 1926),
(1120302, 30424, 'CWA', 'Mosinee, WI', 'WI', '55', 'Wisconsin', '45', 1823),
(1163703, 31637, 'FAR', 'Fargo, ND', 'ND', '38', 'North Dakota', '66', 1421),
(1535602, 35356, 'TTN', 'Trenton, NJ', 'NJ', '34', 'New Jersey', '21', 1050),
(1192102, 31921, 'GJT', 'Grand Junction, CO', 'CO', '8', 'Colorado', '82', 1837),
(1348502, 33485, 'MSN', 'Madison, WI', 'WI', '55', 'Wisconsin', '45', 1040),
(1467903, 33570, 'SAN', 'San Diego, CA', 'CA', '6', 'California', '91', 907),
(1114603, 31146, 'CRW', 'Charleston/Dunbar, WV', 'WV', '54', 'West Virginia', '39', 1255),
(1349503, 33495, 'MSY', 'New Orleans, LA', 'LA', '22', 'Louisiana', '72', 928),
(1087402, 30647, 'CAK', 'Akron, OH', 'OH', '39', 'Ohio', '44', 1232),
(1383002, 33830, 'OGG', 'Kahului, HI', 'HI', '15', 'Hawaii', '2', 1334),
(1342202, 30562, 'MOB', 'Mobile, AL', 'AL', '1', 'Alabama', '51', 2050),
(1127402, 31274, 'DBQ', 'Dubuque, IA', 'IA', '19', 'Iowa', '61', 1620),
(1484202, 34842, 'SJT', 'San Angelo, TX', 'TX', '48', 'Texas', '74', 2035),
(1071302, 30713, 'BOI', 'Boise, ID', 'ID', '16', 'Idaho', '83', 2105),
(1457604, 34576, 'ROC', 'Rochester, NY', 'NY', '36', 'New York', '22', 1850),
(1188402, 31884, 'GEG', 'Spokane, WA', 'WA', '53', 'Washington', '93', 1209),
(1483103, 32457, 'SJC', 'San Jose, CA', 'CA', '6', 'California', '91', 1210),
(1080003, 32575, 'BUR', 'Burbank, CA', 'CA', '6', 'California', '91', 1840),
(1154003, 30615, 'ELP', 'El Paso, TX', 'TX', '48', 'Texas', '74', 1850),
(1389101, 32575, 'ONT', 'Ontario, CA', 'CA', '6', 'California', '91', 1815),
(1457002, 34570, 'RNO', 'Reno, NV', 'NV', '32', 'Nevada', '85', 745),
(1490803, 32575, 'SNA', 'Santa Ana, CA', 'CA', '6', 'California', '91', 1830),
(1042302, 30423, 'AUS', 'Austin, TX', 'TX', '48', 'Texas', '74', 1445),
(1025702, 30257, 'ALB', 'Albany, NY', 'NY', '36', 'New York', '22', 1310),
(1295402, 32575, 'LGB', 'Long Beach, CA', 'CA', '6', 'California', '91', 840),
(1473003, 33044, 'SDF', 'Louisville, KY', 'KY', '21', 'Kentucky', '52', 1825),
(1142303, 31423, 'DSM', 'Des Moines, IA', 'IA', '19', 'Iowa', '61', 2005),
(1489302, 33192, 'SMF', 'Sacramento, CA', 'CA', '6', 'California', '91', 1145),
(1198202, 31982, 'GRK', 'Killeen, TX', 'TX', '48', 'Texas', '74', 1300),
(1163805, 31638, 'FAT', 'Fresno, CA', 'CA', '6', 'California', '91', 1955),
(1621801, 33785, 'YUM', 'Yuma, AZ', 'AZ', '4', 'Arizona', '81', 820),
(1537602, 30436, 'TUS', 'Tucson, AZ', 'AZ', '4', 'Arizona', '81', 1115),
(1426202, 34262, 'PSP', 'Palm Springs, CA', 'CA', '6', 'California', '91', 1735),
(1244102, 32441, 'JAC', 'Jackson, WY', 'WY', '56', 'Wyoming', '88', 750),
(1040802, 30408, 'ATW', 'Appleton, WI', 'WI', '55', 'Wisconsin', '45', 1555),
(1160302, 31603, 'EUG', 'Eugene, OR', 'OR', '41', 'Oregon', '92', 2110),
(1215602, 32156, 'HLN', 'Helena, MT', 'MT', '30', 'Montana', '84', 2012),
(1504102, 35041, 'SUN', 'Sun Valley/Hailey/Ketchum, ID', 'ID', '16', 'Idaho', '83', 1717),
(1238902, 32389, 'ISN', 'Williston, ND', 'ND', '38', 'North Dakota', '66', 1135),
(1315802, 33158, 'MAF', 'Midland/Odessa, TX', 'TX', '48', 'Texas', '74', 942),
(1469802, 34236, 'SBP', 'San Luis Obispo, CA', 'CA', '6', 'California', '91', 1552),
(1495203, 34952, 'SPI', 'Springfield, IL', 'IL', '17', 'Illinois', '41', 1100),
(1347603, 34922, 'MRY', 'Monterey, CA', 'CA', '6', 'California', '91', 743),
(1037203, 30372, 'ASE', 'Aspen, CO', 'CO', '8', 'Colorado', '82', 1817),
(1084903, 30849, 'BZN', 'Bozeman, MT', 'MT', '30', 'Montana', '84', 1710),
(1015502, 30155, 'ACT', 'Waco, TX', 'TX', '48', 'Texas', '74', 1725),
(1110902, 30189, 'COS', 'Colorado Springs, CO', 'CO', '8', 'Colorado', '82', 1817),
(1468902, 34689, 'SBA', 'Santa Barbara, CA', 'CA', '6', 'California', '91', 1529),
(1015703, 30157, 'ACV', 'Arcata/Eureka, CA', 'CA', '6', 'California', '91', 744),
(1029904, 30299, 'ANC', 'Anchorage, AK', 'AK', '2', 'Alaska', '1', 1918),
(1062002, 30620, 'BIL', 'Billings, MT', 'MT', '30', 'Montana', '84', 2214),
(1457402, 34574, 'ROA', 'Roanoke, VA', 'VA', '51', 'Virginia', '38', 2040),
(1484304, 34819, 'SJU', 'San Juan, PR', 'PR', '72', 'Puerto Rico', '3', 1005),
(1148102, 31481, 'ECP', 'Panama City, FL', 'FL', '12', 'Florida', '33', 1646),
(1562401, 31504, 'VPS', 'Valparaiso, FL', 'FL', '12', 'Florida', '33', 1340),
(1219702, 31703, 'HPN', 'White Plains, NY', 'NY', '36', 'New York', '22', 705),
(1078502, 30785, 'BTV', 'Burlington, VT', 'VT', '50', 'Vermont', '16', 1559),
(1348602, 33486, 'MSO', 'Missoula, MT', 'MT', '30', 'Montana', '84', 2146),
(1419304, 33728, 'PNS', 'Pensacola, FL', 'FL', '12', 'Florida', '33', 2144),
(1114004, 31140, 'CRP', 'Corpus Christi, TX', 'TX', '48', 'Texas', '74', 1420),
(1289605, 32896, 'LBB', 'Lubbock, TX', 'TX', '48', 'Texas', '74', 1030),
(1240203, 32402, 'ITO', 'Hilo, HI', 'HI', '15', 'Hawaii', '2', 1455),
(1228002, 32280, 'IDA', 'Idaho Falls, ID', 'ID', '16', 'Idaho', '83', 2122),
(1415002, 34150, 'PLN', 'Pellston, MI', 'MI', '26', 'Michigan', '43', 1202),
(1326403, 33264, 'MFR', 'Medford, OR', 'OR', '41', 'Oregon', '92', 1703),
(1445702, 34457, 'RAP', 'Rapid City, SD', 'SD', '46', 'South Dakota', '67', 1510),
(1133703, 31337, 'DLH', 'Duluth, MN', 'MN', '27', 'Minnesota', '63', 845),
(1073203, 30732, 'BQN', 'Aguadilla, PR', 'PR', '72', 'Puerto Rico', '3', 2052),
(1275803, 32758, 'KOA', 'Kona, HI', 'HI', '15', 'Hawaii', '2', 1300),
(1078102, 30781, 'BTR', 'Baton Rouge, LA', 'LA', '22', 'Louisiana', '72', 1856),
(1469606, 34696, 'SBN', 'South Bend, IN', 'IN', '18', 'Indiana', '42', 1754),
(1046902, 30469, 'AZO', 'Kalamazoo, MI', 'MI', '26', 'Michigan', '43', 1645),
(1162402, 31624, 'EYW', 'Key West, FL', 'FL', '12', 'Florida', '33', 800),
(1043403, 30434, 'AVP', 'Scranton/Wilkes-Barre, PA', 'PA', '42', 'Pennsylvania', '23', 1444),
(1298202, 32982, 'LIH', 'Lihue, HI', 'HI', '15', 'Hawaii', '2', 905),
(1251102, 32511, 'JLN', 'Joplin, MO', 'MO', '29', 'Missouri', '64', 1930),
(1524903, 35249, 'TLH', 'Tallahassee, FL', 'FL', '12', 'Florida', '33', 1650),
(1336003, 33360, 'MLB', 'Melbourne, FL', 'FL', '12', 'Florida', '33', 1506),
(1379603, 32457, 'OAK', 'Oakland, CA', 'CA', '6', 'California', '91', 920),
(1591902, 31834, 'XNA', 'Fayetteville, AR', 'AR', '5', 'Arkansas', '71', 910),
(1101303, 31013, 'CIU', 'Sault Ste. Marie, MI', 'MI', '26', 'Michigan', '43', 1200),
(1152502, 31525, 'EKO', 'Elko, NV', 'NV', '32', 'Nevada', '85', 2156),
(1125203, 31252, 'DAB', 'Daytona Beach, FL', 'FL', '12', 'Florida', '33', 1035),
(1560702, 35550, 'VLD', 'Valdosta, GA', 'GA', '13', 'Georgia', '34', 1113),
(1432103, 34321, 'PWM', 'Portland, ME', 'ME', '23', 'Maine', '12', 740),
(1104103, 31041, 'CLD', 'Carlsbad, CA', 'CA', '6', 'California', '91', 1528),
(1532302, 35323, 'TRI', 'Bristol/Johnson City/Kingsport, TN', 'TN', '47', 'Tennessee', '54', 1210),
(1186502, 31865, 'GCC', 'Gillette, WY', 'WY', '56', 'Wyoming', '88', 1915),
(1093002, 30930, 'CEC', 'Crescent City, CA', 'CA', '6', 'California', '91', 840),
(1200302, 31895, 'GTF', 'Great Falls, MT', 'MT', '30', 'Montana', '84', 1130),
(1073905, 30739, 'BRD', 'Brainerd, MN', 'MN', '27', 'Minnesota', '63', 1431),
(1217703, 32177, 'HOB', 'Hobbs, NM', 'NM', '35', 'New Mexico', '86', 1055),
(1197302, 31973, 'GPT', 'Gulfport/Biloxi, MS', 'MS', '28', 'Mississippi', '53', 1345),
(1325602, 33256, 'MFE', 'Mission/McAllen/Edinburg, TX', 'TX', '48', 'Texas', '74', 2112),
(1195302, 31953, 'GNV', 'Gainesville, FL', 'FL', '12', 'Florida', '33', 1258),
(1099002, 30990, 'CHO', 'Charlottesville, VA', 'VA', '51', 'Virginia', '38', 1030),
(1209402, 34699, 'HDN', 'Hayden, CO', 'CO', '8', 'Colorado', '82', 1005),
(1549703, 35497, 'UST', 'St. Augustine, FL', 'FL', '12', 'Florida', '33', 1510),
(1329002, 33290, 'MHK', 'Manhattan/Ft. Riley, KS', 'KS', '20', 'Kansas', '62', 1055),
(1114003, 31140, 'CRP', 'Corpus Christi, TX', 'TX', '48', 'Texas', '74', 1120),
(1343302, 33304, 'MOT', 'Minot, ND', 'ND', '38', 'North Dakota', '66', 1925),
(1419303, 33728, 'PNS', 'Pensacola, FL', 'FL', '12', 'Florida', '33', 1005),
(1098002, 30980, 'CHA', 'Chattanooga, TN', 'TN', '47', 'Tennessee', '54', 1604),
(1232303, 32323, 'ILM', 'Wilmington, NC', 'NC', '37', 'North Carolina', '36', 1730),
(1130802, 31308, 'DHN', 'Dothan, AL', 'AL', '1', 'Alabama', '51', 1635),
(1112202, 31122, 'CPR', 'Casper, WY', 'WY', '56', 'Wyoming', '88', 805),
(1334402, 33344, 'MKG', 'Muskegon, MI', 'MI', '26', 'Michigan', '43', 951),
(1448902, 34489, 'RDM', 'Bend/Redmond, OR', 'OR', '41', 'Oregon', '92', 1231),
(1064301, 30643, 'BKG', 'Branson, MO', 'MO', '29', 'Missouri', '64', 1010),
(1220603, 32206, 'HRL', 'Harlingen/San Benito, TX', 'TX', '48', 'Texas', '74', 1700),
(1288403, 32884, 'LAN', 'Lansing, MI', 'MI', '26', 'Michigan', '43', 2058),
(1172102, 31721, 'FNT', 'Flint, MI', 'MI', '26', 'Michigan', '43', 1459),
(1540103, 35401, 'TXK', 'Texarkana, AR', 'AR', '5', 'Arkansas', '71', 1020),
(1425202, 34252, 'PSC', 'Pasco/Kennewick/Richland, WA', 'WA', '53', 'Washington', '93', 1124),
(1342402, 33424, 'MOD', 'Modesto, CA', 'CA', '6', 'California', '91', 959),
(1538902, 35389, 'TWF', 'Twin Falls, ID', 'ID', '16', 'Idaho', '83', 1119),
(1015403, 30154, 'ACK', 'Nantucket, MA', 'MA', '25', 'Massachusetts', '13', 1310),
(1411302, 34113, 'PIH', 'Pocatello, ID', 'ID', '16', 'Idaho', '83', 1104),
(1448702, 33792, 'RDD', 'Redding, CA', 'CA', '6', 'California', '91', 1859),
(1541103, 35411, 'TYR', 'Tyler, TX', 'TX', '48', 'Texas', '74', 818),
(1153703, 31537, 'ELM', 'Elmira/Corning, NY', 'NY', '36', 'New York', '22', 1354),
(1104902, 31049, 'CLL', 'College Station/Bryan, TX', 'TX', '48', 'Texas', '74', 1156),
(1337702, 33377, 'MLU', 'Monroe, LA', 'LA', '22', 'Louisiana', '72', 1055),
(1538003, 35380, 'TVC', 'Traverse City, MI', 'MI', '26', 'Michigan', '43', 1550),
(1189802, 31898, 'GFK', 'Grand Forks, ND', 'ND', '38', 'North Dakota', '66', 1600),
(1112203, 31122, 'CPR', 'Casper, WY', 'WY', '56', 'Wyoming', '88', 1711),
(1454302, 34543, 'RKS', 'Rock Springs, WY', 'WY', '56', 'Wyoming', '88', 1710),
(1379502, 33795, 'OAJ', 'Jacksonville/Camp Lejeune, NC', 'NC', '37', 'North Carolina', '36', 2130),
(1131502, 31315, 'DIK', 'Dickinson, ND', 'ND', '38', 'North Dakota', '66', 1815),
(1164102, 31641, 'FAY', 'Fayetteville, NC', 'NC', '37', 'North Carolina', '36', 1502),
(1402501, 34025, 'PBG', 'Plattsburgh, NY', 'NY', '36', 'New York', '22', 2149),
(1157704, 31577, 'ERI', 'Erie, PA', 'PA', '42', 'Pennsylvania', '23', 1927),
(1014602, 30146, 'ABY', 'Albany, GA', 'GA', '13', 'Georgia', '34', 1359),
(1013503, 30135, 'ABE', 'Allentown/Bethlehem/Easton, PA', 'PA', '42', 'Pennsylvania', '23', 1955),
(1129804, 30194, 'DFW', 'Dallas/Fort Worth, TX', 'TX', '48', 'Texas', '74',930),
(1247803, 31703, 'JFK', 'New York, NY', 'NY', '36', 'New York', '22', 2130),
(1016503, 30165, 'ADK', 'Adak Island, AK', 'AK', '2', 'Alaska', '1', 1450),
(1393303, 33933, 'ORH', 'Worcester, MA', 'MA', '25', 'Massachusetts', '13', 1344),
(1426204, 34262, 'PSP', 'Palm Springs, CA', 'CA', '6', 'California', '91', 112,2),
(1467402, 34674, 'SAF', 'Santa Fe, NM', 'NM', '35', 'New Mexico', '86', 850),
(1524904, 35249, 'TLH', 'Tallahassee, FL', 'FL', '12', 'Florida', '33', 1230),
(1295104, 32951, 'LFT', 'Lafayette, LA', 'LA', '22', 'Louisiana', '72', 2110),
(1057703, 30577, 'BGM', 'Binghamton, NY', 'NY', '36', 'New York', '22', 1526),
(1068502, 30685, 'BMI', 'Bloomington/Normal, IL', 'IL', '17', 'Illinois', '41', 1503),
(1073103, 30731, 'BQK', 'Brunswick, GA', 'GA', '13', 'Georgia', '34', 2005),
(1471102, 34711, 'SCE', 'State College, PA', 'PA', '42', 'Pennsylvania', '23', 1810),
(1336703, 33367, 'MLI', 'Moline, IL', 'IL', '17', 'Illinois', '41', 1737),
(1463303, 32547, 'RST', 'Rochester, MN', 'MN', '27', 'Minnesota', '63', 2044),
(1144703, 31447, 'DVL', 'Devils Lake, ND', 'ND', '38', 'North Dakota', '66', 1343),
(1147103, 31471, 'EAU', 'Eau Claire, WI', 'WI', '55', 'Wisconsin', '45', 905),
(1234303, 32343, 'INL', 'International Falls, MN', 'MN', '27', 'Minnesota', '63', 1116),
(1397003, 33970, 'OTZ', 'Kotzebue, AK', 'AK', '2', 'Alaska', '1', 1905),
(1562402, 31504, 'VPS', 'Valparaiso, FL', 'FL', '12', 'Florida', '33', 1344),
(1074702, 30747, 'BRO', 'Brownsville, TX', 'TX', '48', 'Texas', '74', 1820),
(1078103, 30781, 'BTR', 'Baton Rouge, LA', 'LA', '22', 'Louisiana', '72', 2305),
(1018502, 30185, 'AEX', 'Alexandria, LA', 'LA', '22', 'Louisiana', '72', 1001),
(1302902, 33029, 'LNK', 'Lincoln, NE', 'NE', '31', 'Nebraska', '65', 1910),
(1504803, 35048, 'SUX', 'Sioux City, IA', 'IA', '19', 'Iowa', '61', 2041),
(1013603, 30136, 'ABI', 'Abilene, TX', 'TX', '48', 'Texas', '74', 2350),
(1141304, 30285, 'DRO', 'Durango, CO', 'CO', '8', 'Colorado', '82', 954),
(1307603, 33076, 'LSE', 'La Crosse, WI', 'WI', '55', 'Wisconsin', '45', 1725),
(1507002, 31703, 'SWF', 'Newburgh/Poughkeepsie, NY', 'NY', '36', 'New York', '22', 1915),
(1227803, 30928, 'ICT', 'Wichita, KS', 'KS', '20', 'Kansas', '62', 1555),
(1111102, 32474, 'COU', 'Columbia, MO', 'MO', '29', 'Missouri', '64', 1415),
(1541203, 35412, 'TYS', 'Knoxville, TN', 'TN', '47', 'Tennessee', '54', 1819),
(1479403, 34794, 'SGU', 'St. George, UT', 'UT', '49', 'Utah', '87', 1121),
(1040803, 30408, 'ATW', 'Appleton, WI', 'WI', '55', 'Wisconsin', '45', 1554),
(1457403, 34574, 'ROA', 'Roanoke, VA', 'VA', '51', 'Virginia', '38', 1854),
(1177801, 31778, 'FSM', 'Fort Smith, AR', 'AR', '5', 'Arkansas', '71', 2112),
(1043103, 30431, 'AVL', 'Asheville, NC', 'NC', '37', 'North Carolina', '36', 1535),
(1234302, 32343, 'INL', 'International Falls, MN', 'MN', '27', 'Minnesota', '63', 2152),
(1496002, 34960, 'SPS', 'Wichita Falls, TX', 'TX', '48', 'Texas', '74', 850),
(1324102, 33241, 'MEI', 'Meridian, MS', 'MS', '28', 'Mississippi', '53', 1215),
(1318403, 33184, 'MBS', 'Saginaw/Bay City/Midland, MI', 'MI', '26', 'Michigan', '43', 645),
(1400602, 34006, 'PAH', 'Paducah, KY', 'KY', '21', 'Kentucky', '52', 1414),
(1239702, 32397, 'ITH', 'Ithaca/Cortland, NY', 'NY', '36', 'New York', '22', 1955),
(1422204, 34222, 'PPG', 'Pago Pago, TT', 'TT', '75', 'U.S. Pacific Trust Territories and Possessions', '5', 12173),
(1186703, 31867, 'GCK', 'Garden City, KS', 'KS', '20', 'Kansas', '62', 11298),
(1161203, 31612, 'EVV', 'Evansville, IN', 'IN', '18', 'Indiana', '42', 13930),
(1215603, 32156, 'HLN', 'Helena, MT', 'MT', '30', 'Montana', '84', 11292),
(1062702, 30627, 'BIS', 'Bismarck/Mandan, ND', 'ND', '38', 'North Dakota', '66', 1327),
(1164802, 31648, 'FCA', 'Kalispell, MT', 'MT', '30', 'Montana', '84', 950),
(1169502, 31695, 'FLG', 'Flagstaff, AZ', 'AZ', '4', 'Arizona', '81', 1615),
(1428803, 34288, 'PUB', 'Pueblo, CO', 'CO', '8', 'Colorado', '82', 1541),
(1107602, 31076, 'CMX', 'Hancock/Houghton, MI', 'MI', '26', 'Michigan', '43', 1900),
(1077902, 30779, 'BTM', 'Butte, MT', 'MT', '30', 'Montana', '84', 1713),
(1410902, 34109, 'PIB', 'Hattiesburg/Laurel, MS', 'MS', '28', 'Mississippi', '53', 2055),
(1212903, 32129, 'HIB', 'Hibbing, MN', 'MN', '27', 'Minnesota', '63', 2000),
(1469605, 34696, 'SBN', 'South Bend, IN', 'IN', '18', 'Indiana', '42', 1950),
(1036102, 30361, 'ART', 'Watertown, NY', 'NY', '36', 'New York', '22', 1730),
(1482803, 34828, 'SIT', 'Sitka, AK', 'AK', '2', 'Alaska', '1', 830),
(1100202, 31002, 'CIC', 'Chico, CA', 'CA', '6', 'California', '91', 1130),
(1306104, 33038, 'LRD', 'Laredo, TX', 'TX', '48', 'Texas', '74', 1515),
(1014103, 30141, 'ABR', 'Aberdeen, SD', 'SD', '46', 'South Dakota', '67', 2005),
(1289803, 32898, 'LBE', 'Latrobe, PA', 'PA', '42', 'Pennsylvania', '23', 1455),
(1502704, 34992, 'STX', 'Christiansted, VI', 'VI', '78', 'U.S. Virgin Islands', '4', 1215),
(1114005, 31140, 'CRP', 'Corpus Christi, TX', 'TX', '48', 'Texas', '74', 1910),
(1251902, 32519, 'JMS', 'Jamestown, ND', 'ND', '38', 'North Dakota', '66', 1045),
(1425403, 34254, 'PSE', 'Ponce, PR', 'PR', '72', 'Puerto Rico', '3', 2203),
(1152503, 31525, 'EKO', 'Elko, NV', 'NV', '32', 'Nevada', '85', 2153),
(1252302, 32523, 'JNU', 'Juneau, AK', 'AK', '2', 'Alaska', '1', 14747),
(1072804, 30728, 'BPT', 'Beaumont/Port Arthur, TX', 'TX', '48', 'Texas', '74', 11298),
(1161706, 31617, 'EWN', 'New Bern/Morehead/Beaufort, NC', 'NC', '37', 'North Carolina', '36', 1035),
(1058102, 30581, 'BGR', 'Bangor, ME', 'ME', '23', 'Maine', '12', 1350),
(1198002, 31980, 'GRI', 'Grand Island, NE', 'NE', '31', 'Nebraska', '65', 1300),
(1033302, 30333, 'APN', 'Alpena, MI', 'MI', '26', 'Michigan', '43', 1553),
(1452002, 34520, 'RHI', 'Rhinelander, WI', 'WI', '55', 'Wisconsin', '45', 1415),
(1558202, 35582, 'VEL', 'Vernal, UT', 'UT', '49', 'Utah', '87', 2152),
(1200702, 30894, 'GTR', 'Columbus, MS', 'MS', '28', 'Mississippi', '53', 1255),
(1307602, 33076, 'LSE', 'La Crosse, WI', 'WI', '55', 'Wisconsin', '45', 1945),
(1106702, 31067, 'CMI', 'Champaign/Urbana, IL', 'IL', '17', 'Illinois', '41', 1950),
(1387302, 33873, 'OME', 'Nome, AK', 'AK', '2', 'Alaska', '1', 1025),
(1066602, 30666, 'BLI', 'Bellingham, WA', 'WA', '53', 'Washington', '93', 2340),
(1289102, 32891, 'LAW', 'Lawton/Fort Sill, OK', 'OK', '40', 'Oklahoma', '73', 1615),
(1225502, 32255, 'HYS', 'Hays, KS', 'KS', '20', 'Kansas', '62', 1715),
(1500802, 32556, 'STC', 'St. Cloud, MN', 'MN', '27', 'Minnesota', '63', 1820),
(1239102, 31703, 'ISP', 'Islip, NY', 'NY', '36', 'New York', '22', 710),
(1479402, 34794, 'SGU', 'St. George, UT', 'UT', '49', 'Utah', '87', 1112),
(1490503, 34905, 'SMX', 'Santa Maria, CA', 'CA', '6', 'California', '91', 1239),
(1055102, 30113, 'BET', 'Bethel, AK', 'AK', '2', 'Alaska', '1', 1140),
(1133602, 31336, 'DLG', 'Dillingham, AK', 'AK', '2', 'Alaska', '1', 1040),
(1312702, 33127, 'LWS', 'Lewiston, ID', 'ID', '16', 'Idaho', '83', 2151),
(1172602, 31726, 'FOE', 'Topeka, KS', 'KS', '20', 'Kansas', '62', 2046),
(1252303, 32523, 'JNU', 'Juneau, AK', 'AK', '2', 'Alaska', '1', 610),
(1016502, 30165, 'ADK', 'Adak Island, AK', 'AK', '2', 'Alaska', '1', 1458),
(1201202, 32012, 'GUC', 'Gunnison, CO', 'CO', '8', 'Colorado', '82', 1106),
(1345902, 33459, 'MQT', 'Marquette, MI', 'MI', '26', 'Michigan', '43', 1544),
(1201203, 32012, 'GUC', 'Gunnison, CO', 'CO', '8', 'Colorado', '82', 950),
(1161204, 31612, 'EVV', 'Evansville, IN', 'IN', '18', 'Indiana', '42', 1639),
(1302402, 33024, 'LMT', 'Klamath Falls, OR', 'OR', '41', 'Oregon', '92', 14057),
(1470903, 30073, 'SCC', 'Deadhorse, AK', 'AK', '2', 'Alaska', '1', 1430),
(1281902, 31401, 'KTN', 'Ketchikan, AK', 'AK', '2', 'Alaska', '1', 1130),
(1425603, 34256, 'PSG', 'Petersburg, AK', 'AK', '2', 'Alaska', '1', 1054),
(1026802, 30268, 'ALO', 'Waterloo, IA', 'IA', '19', 'Iowa', '61', 1945),
(1014102, 30141, 'ABR', 'Aberdeen, SD', 'SD', '46', 'South Dakota', '67', 1305),
(1190502, 31905, 'GGG', 'Longview, TX', 'TX', '48', 'Texas', '74', 1700),
(1075403, 30107, 'BRW', 'Barrow, AK', 'AK', '2', 'Alaska', '1', 1735),
(1063104, 30631, 'BJI', 'Bemidji, MN', 'MN', '27', 'Minnesota', '63', 1430),
(1172103, 31721, 'FNT', 'Flint, MI', 'MI', '26', 'Michigan', '43', 1855),
(1599102, 35991, 'YAK', 'Yakutat, AK', 'AK', '2', 'Alaska', '1', 1703),
(1091802, 30918, 'CDC', 'Cedar City, UT', 'UT', '49', 'Utah', '87', 1500),
(1150303, 31503, 'EGE', 'Eagle, CO', 'CO', '8', 'Colorado', '82', 1212),
(1397002, 33970, 'OTZ', 'Kotzebue, AK', 'AK', '2', 'Alaska', '1', 1020),
(1233502, 32335, 'IMT', 'Iron Mountain/Kingsfd, MI', 'MI', '26', 'Michigan', '43', 1110),
(1529502, 35165, 'TOL', 'Toledo, OH', 'OH', '39', 'Ohio', '44', 1115),
(1158702, 31587, 'ESC', 'Escanaba, MI', 'MI', '26', 'Michigan', '43', 1200),
(1201602, 32016, 'GUM', 'Guam, TT', 'TT', '75', 'U.S. Pacific Trust Territories and Possessions', '5', 1405),
(1226503, 32265, 'IAG', 'Niagara Falls, NY', 'NY', '36', 'New York', '22', 2215),
(1288802, 32888, 'LAR', 'Laramie, WY', 'WY', '56', 'Wyoming', '88', 1121),
(1109702, 31097, 'COD', 'Cody, WY', 'WY', '56', 'Wyoming', '88', 2000),
(1109202, 31092, 'CNY', 'Moab, UT', 'UT', '49', 'Utah', '87', 1106),
(1387303, 33873, 'OME', 'Nome, AK', 'AK', '2', 'Alaska', '1', 1005),
(1199702, 31997, 'GST', 'Gustavus, AK', 'AK', '2', 'Alaska', '1', 12523),
(1075402, 30107, 'BRW', 'Barrow, AK', 'AK', '2', 'Alaska', '1', 1721),
(1458801, 34588, 'ROW', 'Roswell, NM', 'NM', '35', 'New Mexico', '86', 1415),
(1252304, 32523, 'JNU', 'Juneau, AK', 'AK', '2', 'Alaska', '1', 14828),
(1092603, 30913, 'CDV', 'Cordova, AK', 'AK', '2', 'Alaska', '1', 1207),
(1584102, 35841, 'WRG', 'Wrangell, AK', 'AK', '2', 'Alaska', '1', 1525),
(1024502, 30245, 'AKN', 'King Salmon, AK', 'AK', '2', 'Alaska', '1', 1545),
(1482802, 34828, 'SIT', 'Sitka, AK', 'AK', '2', 'Alaska', '1', 1015),
(1225002, 32250, 'HYA', 'Hyannis, MA', 'MA', '25', 'Massachusetts', '13', 12478),
(1354102, 33541, 'MVY', "Martha's Vineyard, MA", 'MA', '25', 'Massachusetts', '13', 1329),
(1017001, 30070, 'ADQ', 'Kodiak, AK', 'AK', '2', 'Alaska', '1', 620),
(1589702, 35897, 'WYS', 'West Yellowstone, MT', 'MT', '30', 'Montana', '84', 1345),
(1495503, 34955, 'SPN', 'Saipan, TT', 'TT', '75', 'U.S. Pacific Trust Territories and Possessions', '5', 615),
(1320302, 33155, 'MCN', 'Macon, GA', 'GA', '13', 'Georgia', '34', 10397)]

In [ ]:
resultado = {fila[3]: fila[0] for fila in datos}

print(resultado)
